In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import platform
import matplotlib as mpl

def set_chinese_font():
    """
    Configure matplotlib to properly display Chinese characters on both macOS and Windows.
    """
    system = platform.system()
    
    if system == 'Windows':
        font_list = ['SimHei', 'Microsoft YaHei']
    elif system == 'Darwin':  # macOS
        font_list = ['PingFang HK', 'Arial Unicode MS', 'Heiti TC']
    else:  # Linux or other systems
        font_list = ['WenQuanYi Micro Hei', 'Droid Sans Fallback']
    
    # Try different fonts until one works
    for font in font_list:
        try:
            plt.rcParams['font.sans-serif'] = [font]
            # Test if the font works
            mpl.font_manager.findfont(font)

            print(f"检测到当前系统为: {system}, 已为 matplotlib 设置字体: {font}")
            break
        except:
            continue

set_chinese_font()

In [ ]:
# 读取模拟数据
import pandas as pd
import numpy as np
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')
df

# Law Article 4

## MEU_4_1


| 字段 | 内容 |
|------|------|
| subject | 上市公司大股东 \| 董监高 |
| condition | 计划通过本所集中竞价或大宗交易减持股份 |
| constrain | 应当及时通知公司，并在首次卖出的15个交易日前向本所报告并预先披露减持计划 |
| contextual_info | nan |
| note | 不考虑是否及时通知公司 |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1725 |
| completion_tokens | 7141 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
def check_meu_4_1(df):
    '''
    检查MEU_4_1的合规性：
    主体：上市公司大股东 | 董监高
    条件：存在通过竞价/大宗交易的减持计划
    约束：首次卖出前至少15个交易日披露
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_4_1_subject'] = False
    df['meu_4_1_condition'] = False
    df['meu_4_1_constraint'] = None

    # 1. 验证责任主体（与4.8相同逻辑）
    is_major = (
        df['股东身份'].isin(['控股股东', '实际控制人', '持股5%以上股东']) |
        (df['持股比例'] >= 0.05)
    )
    is_director = df['股东身份'] == '董监高'
    df.loc[is_major | is_director, 'meu_4_1_subject'] = True

    # 2. 验证触发条件
    valid_condition = (
        df['减持方式'].isin(['竞价交易', '大宗交易']) &
        df['存在减持计划']
    )
    df.loc[valid_condition, 'meu_4_1_condition'] = True

    # 3. 核心改进：准确识别首次卖出日
    def process_group(group):
        group = group.sort_values('日期')
        plans = []
        current_plan = None
        
        # 识别减持计划区间
        for _, row in group.iterrows():
            if row['存在减持计划']:
                if current_plan is None:
                    current_plan = {
                        '披露日': row['计划披露日'],
                        '开始日': row['计划开始日'],
                        '减持记录': []
                    }
                if row['当日减持比例'] > 0:
                    current_plan['减持记录'].append(row['日期'])
            else:
                if current_plan:
                    plans.append(current_plan)
                    current_plan = None
        if current_plan:
            plans.append(current_plan)

        # 提取首次卖出日
        return pd.DataFrame([{
            '公司简称': group['公司简称'].iloc[0],
            '股东': group['股东'].iloc[0],
            '计划披露日': p['披露日'],
            '首次卖出日': min(p['减持记录']) if p['减持记录'] else pd.NaT
        } for p in plans])

    # 按分组处理并合并
    plan_info = df.groupby(['公司简称', '股东'], group_keys=False).apply(process_group)
    df = df.merge(
        plan_info,
        on=['公司简称', '股东', '计划披露日'],
        how='left'
    ) if not plan_info.empty else df.assign(首次卖出日=pd.NaT)

    # 4. 交易日差计算（改进点：使用首次卖出日而非计划开始日）
    trading_dates = pd.Series(df['日期'].unique()).sort_values().reset_index(drop=True)
    date_map = {date: idx for idx, date in enumerate(trading_dates)}
    
    df['交易日差'] = df.apply(
        lambda x: date_map.get(x['首次卖出日'], np.nan) - date_map.get(x['计划披露日'], np.nan),
        axis=1
    )

    # 5. 验证约束条件
    # 至少提前15个交易日披露
    df['meu_4_1_constraint'] = (
        df['首次卖出日'].isna() |  # 未卖出视为合规
        (df['交易日差'] >= 15)     # 或提前足够天数披露
    )

    return df

In [ ]:
df = check_meu_4_1(df)
df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def visualize_meu_4_1_results(df, show_groups=None):
    # 按公司简称和股东分组
    grouped = list(df.groupby(['公司简称', '股东']))
    
    # 如果指定了show_groups，限制展示的组数
    if show_groups is not None:
        grouped = grouped[:min(show_groups, len(grouped))]
    
    for (company, shareholder), group in grouped:
        plt.figure(figsize=(15, 8))
        
        # 折线图：持股比例变化
        plt.plot(group['日期'], group['持股比例'], 
                 marker='o', markersize=5, label='持股比例')
        
        # 标记股东身份变化
        status_changes = group[group['股东身份'].ne(group['股东身份'].shift())]
        for _, row in status_changes.iterrows():
            plt.axvline(row['日期'], color='gray', linestyle='--', alpha=0.5)
            plt.text(row['日期'], plt.ylim()[1]*0.95, 
                     f"身份变更为\n{row['股东身份']}",
                     rotation=90, va='top', ha='right', fontsize=8)

        # 提取减持计划信息
        plans = group[['计划披露日', '首次卖出日', '交易日差', 'meu_4_1_constraint']
                     ].drop_duplicates().dropna(subset=['计划披露日', 'meu_4_1_constraint'])

        # 绘制每个减持计划的相关标记
        for _, plan in plans.iterrows():
            # 计划披露日标记（蓝色五角星）
            disclosure_date = plan['计划披露日']
            y_val = group[group['日期'] == disclosure_date]['持股比例'].values[0]
            plt.scatter(disclosure_date, y_val, 
                       color='blue', marker='*', s=200, zorder=5,
                       label='减持计划披露日')

            # 根据constraint确定颜色
            color = 'green' if plan['meu_4_1_constraint'] else 'red'
            
            # 如果有首次卖出日，绘制标记和参考线
            if not pd.isna(plan['首次卖出日']):
                sell_date = plan['首次卖出日']
                y_sell = group[group['日期'] == sell_date]['持股比例'].values[0]
                plt.scatter(sell_date, y_sell,
                           color=color, marker='X', s=150, zorder=5,
                           label='首次卖出日' if _ == 0 else "")
                
                # 绘制参考线（披露日后15个交易日）
                delta = plan['交易日差']
                plt.axvspan(disclosure_date, sell_date, 
                           alpha=0.2, color=color)
                plt.text(sell_date, y_sell*0.8,
                        f"间隔{int(delta)}天",
                        rotation=90, fontsize=8, color=color)
            else:
                # 如果没有首次卖出日，只在披露日旁边添加颜色标记
                plt.text(disclosure_date, y_val*0.9,
                        "无卖出日" if _ == 0 else "",
                        rotation=90, fontsize=8, color=color)
                plt.scatter(disclosure_date, y_val*0.95,
                           color=color, marker='s', s=50, zorder=5)

        # 图表装饰
        plt.title(f"{company} - {shareholder}\n持股变动及减持计划验证")
        plt.xlabel('日期')
        plt.ylabel('持股比例')
        plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        
        # 处理图例去重
        handles, labels = plt.gca().get_legend_handles_labels()
        seen = []
        unique = [(h, l) for h, l in zip(handles, labels) if l not in seen and not seen.append(l)]
        plt.legend(*zip(*unique), loc='upper right')
        
        plt.tight_layout()
        plt.show()

visualize_meu_4_1_results(df, show_groups=10)

---

## MEU_4_7


| 字段 | 内容 |
|------|------|
| subject | 上市公司大股东 \| 董监高 |
| condition | 披露减持计划 |
| constrain | 每次披露的减持计划中减持时间区间不得超过3个月 |
| contextual_info | nan |
| note | 三个月按照90个自然日计算 |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1709 |
| completion_tokens | 5009 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_4_7(df):
    '''
    验证MEU_4_7合规性：
    - subject: 上市公司大股东或董监高
    - condition: 披露减持计划
    - constraint: 减持时间区间不超过90自然日
    '''
    df = df.copy()

    # 初始化合规性标记列
    df['meu_4_7_subject'] = False
    df['meu_4_7_condition'] = False
    df['meu_4_7_constraint'] = None

    # 1. 验证责任主体
    # 判断大股东（含持股5%以上）或董监高
    is_major_shareholder = (
        df['股东身份'].isin(['控股股东', '实际控制人', '持股5%以上股东']) |
        (df['持股比例'] >= 0.05)
    )
    is_director = df['股东身份'] == '董监高'
    valid_subject = is_major_shareholder | is_director

    # 2. 验证触发条件
    valid_condition = df['存在减持计划']

    # 3. 验证时间区间约束（独立检查）
    # 计算自然日差并判断有效性
    has_valid_dates = df['计划开始日'].notna() & df['计划结束日'].notna()
    end_after_start = df['计划结束日'] >= df['计划开始日']
    days_diff = (df['计划结束日'] - df['计划开始日']).dt.days
    valid_constraint = has_valid_dates & end_after_start & (days_diff <= 90)

    # 标记各条件满足情况
    df.loc[valid_subject, 'meu_4_7_subject'] = True
    df.loc[valid_condition, 'meu_4_7_condition'] = True
    df.loc[valid_constraint, 'meu_4_7_constraint'] = True
    df.loc[~valid_constraint, 'meu_4_7_constraint'] = False

    return df

In [ ]:
df = check_meu_4_7(df)
# df[(df['存在减持计划'] == False) & (df['计划披露日'].notna())][['日期', '存在减持计划', '计划披露日']]
print('{}, {}, {}'.format(df['meu_4_7_subject'].sum(), df['meu_4_7_condition'].sum(), df['meu_4_7_constraint'].sum()))


In [ ]:
df['减持方式'][0]

---

## MEU_4_8


| 字段 | 内容 |
|------|------|
| subject | 上市公司大股东 \| 董监高 |
| condition | 拟在3个月内通过集中竞价交易减持股份的总数超过公司股份总数1% |
| constrain | 应当在首次卖出的30个交易日前预先披露减持计划 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1717 |
| completion_tokens | 7786 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
def check_meu_4_8(df):
    '''
    检查MEU_4_8合规性：
    subject: 上市公司大股东 | 董监高
    condition: 拟在3个月内通过集中竞价交易减持股份总数超1%
    constraint: 需在首次卖出30个交易日前披露计划
    '''
    df = df.copy()

    # 初始化合规标记列
    df['meu_4_8_subject'] = False
    df['meu_4_8_condition'] = False
    df['meu_4_8_constraint'] = None

    # 1. 验证责任主体 (subject)
    is_major_shareholder = (
        df['股东身份'].isin(['控股股东', '实际控制人', '持股5%以上股东']) |
        (df['持股比例'] >= 0.05)
    )
    is_director = df['股东身份'] == '董监高'
    df.loc[is_major_shareholder | is_director, 'meu_4_8_subject'] = True

    # 2. 验证触发条件 (condition)
    valid_condition = (
        (df['减持方式'] == '竞价交易') &
        df['存在减持计划'] &
        (df['计划减持比例'] > 0.01) &
        ((df['计划结束日'] - df['计划开始日']).dt.days <= 90)
    )
    df.loc[valid_condition.fillna(False), 'meu_4_8_condition'] = True

    # 3. 验证约束条件 (constraint)
    # 生成交易日历映射
    trading_dates = pd.Series(df['日期'].unique()).sort_values().reset_index(drop=True)
    date_to_position = {date: idx for idx, date in enumerate(trading_dates)}

    # 核心处理逻辑
    def process_group(group):
        group = group.sort_values('日期')
        company = group['公司简称'].iloc[0]
        shareholder = group['股东'].iloc[0]
        plans = []
        current_plan = None
        
        # 遍历每个记录识别减持计划区间
        for _, row in group.iterrows():
            if row['存在减持计划']:
                if current_plan is None:
                    # 新计划开始
                    current_plan = {
                        '披露日': row['计划披露日'],
                        '开始日': row['计划开始日'],
                        '结束日': row['计划结束日'],
                        '减持记录': []
                    }
                # 记录减持信息
                if row['当日减持比例'] > 0:
                    current_plan['减持记录'].append(row['日期'])
            else:
                if current_plan is not None:
                    # 计划结束
                    plans.append(current_plan)
                    current_plan = None
        
        # 处理最后一个计划
        if current_plan is not None:
            plans.append(current_plan)

        # 计算每个计划的首次卖出日
        results = []
        for plan in plans:
            first_sell = min(plan['减持记录']) if plan['减持记录'] else pd.NaT
            results.append({
                '公司简称': company,                # 添加分组键
                '股东': shareholder,               # 添加分组键
                '计划披露日': plan['披露日'],
                '首次卖出日': first_sell,
                '计划开始日': plan['开始日'],
                '计划结束日': plan['结束日']
        })
        
        return pd.DataFrame(results)

    # 按公司和股东分组处理
    plan_info = df.groupby(['公司简称', '股东'], group_keys=False).apply(process_group).reset_index(drop=True)

    # 合并计算结果
    if not plan_info.empty:
        df = df.merge(
            plan_info,
            on=['公司简称', '股东', '计划披露日', '计划开始日', '计划结束日'],
            how='left'
        )
    else:
        df['首次卖出日'] = pd.NaT

    # 计算交易日差
    df['交易日差'] = df.apply(
        lambda x: date_to_position.get(x['首次卖出日'], np.nan) - date_to_position.get(x['计划披露日'], np.nan)
        if pd.notnull(x['首次卖出日']) and pd.notnull(x['计划披露日']) 
        else np.nan,
        axis=1
    )

    # 验证约束条件
    # 至少提前30天披露
    df['meu_4_8_constraint'] = (
        df['首次卖出日'].isna() |  # 未卖出视为合规
        (df['交易日差'] >= 30)     # 或提前足够天数披露
    )

    return df

In [ ]:
df = check_meu_4_8(df)
# df[df['交易日差'].notna()][['公司简称', '股东', '计划披露日', '计划开始日', '计划结束日', '首次卖出日', '交易日差']]
print('{}, {}, {}'.format(df['meu_4_8_subject'].sum(), df['meu_4_8_condition'].sum(), df['meu_4_8_constraint'].sum()))

---

# Law Article 9

## MEU_9_1


| 字段 | 内容 |
|------|------|
| subject | 控股股东 \| 实际控制人 \| 董监高 |
| condition | 公司上市时未盈利且处于实现盈利前阶段 |
| constrain | 自公司股票上市之日起2个完整会计年度内不得减持公开发行并上市前股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1721 |
| completion_tokens | 7556 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_9_1(df):
    '''
    检查MEU_9_1合规性：
    subject: 控股股东 | 实际控制人 | 董监高
    condition: 公司上市时未盈利且当前仍未盈利
    constraint: 自上市之日起2个完整会计年度内不得减持
    
    三个条件完全独立检查，不互相干扰
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_9_1_subject'] = False
    df['meu_9_1_condition'] = False
    df['meu_9_1_constraint'] = None

    # ===================== 1. 检查subject =====================
    df['meu_9_1_subject'] = df['股东身份'].isin(['控股股东', '实际控制人', '董监高'])

    # ===================== 2. 检查condition =====================
    # 计算公司上市时净利润
    ipo_profit = df.groupby('公司简称').apply(
        lambda x: x.set_index('日期').loc[x['上市日期'].iloc[0], '净利润']
    )
    df['_temp_ipo_profit'] = df['公司简称'].map(ipo_profit)
    
    # 检查上市至今是否曾盈利（上市后是否曾有净利润>0）
    since_ipo_profited = df.groupby('公司简称').apply(
        lambda x: x[x['日期'] >= x['上市日期'].iloc[0]]['净利润'].gt(0).any()
    )
    df['_temp_since_ipo_profited'] = df['公司简称'].map(since_ipo_profited)
    
    # condition条件：上市时未盈利且上市至今未盈利
    df['meu_9_1_condition'] = (df['_temp_ipo_profit'] <= 0) & (~df['_temp_since_ipo_profited'])

    # ===================== 3. 检查constraint =====================
    # 计算两年后日期
    df['_temp_two_year'] = df['上市日期'] + pd.DateOffset(years=2)
    
    # constraint条件：在上市后2年内不得减持
    in_period = (df['日期'] <= df['_temp_two_year'])
    has_sold = (df['当日减持比例'] > 0)
    df['meu_9_1_constraint'] = ~(in_period & has_sold)

    # 清理临时列
    df.drop(columns=['_temp_ipo_profit', '_temp_two_year'], inplace=True)
    
    return df

In [ ]:
df = check_meu_9_1(df)
df

---

## MEU_9_3


| 字段 | 内容 |
|------|------|
| subject | 董监高 |
| condition | 在前款规定的2个完整会计年度期间内离职 |
| constrain | 应当继续遵守前款规定的减持限制 |
| contextual_info | nan |
| note | nan |
| relation | refer_to |
| target | MEU_9_1 |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1807 |
| completion_tokens | 9305 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
# import pandas as pd

# # 假设df是你的DataFrame
# # 首先检查两列的数据类型
# print("'离任日期'列的数据类型:", df['离任日期'].dtype)
# print("'日期'列的数据类型:", df['日期'].dtype)

# # 检查每行的数据类型是否一致
# # 创建一个新列来标记类型是否一致
# df['类型一致'] = df.apply(lambda row: type(row['离任日期']) == type(row['日期']), axis=1)

# # 找出类型不一致的行
# inconsistent_rows = df[~df['类型一致']]
# print("\n类型不一致的行数:", len(inconsistent_rows))

# if len(inconsistent_rows) > 0:
#     print("\n类型不一致的行:")
#     # 显示不一致行的索引和两列的值及其类型
#     for idx, row in inconsistent_rows.iterrows():
#         print(f"\n行索引: {idx}")
#         print(f"'离任日期': 值={row['离任日期']}, 类型={type(row['离任日期'])}")
#         print(f"'日期': 值={row['日期']}, 类型={type(row['日期'])}")
# else:
#     print("\n所有行的类型都一致")

# # 可选：显示所有行的类型信息
# print("\n所有行的详细类型信息:")
# for idx, row in df.iterrows():
#     print(f"行 {idx}: '离任日期'类型={type(row['离任日期'])}, '日期'类型={type(row['日期'])}")

In [ ]:
import pandas as pd

def check_meu_9_3(df):
    '''
    验证MEU_9_3合规性：
    subject: 董监高
    condition: 在MEU_9_1规定的两年内离职
    constraint: 继续遵守MEU_9_1的规则
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_9_3_subject'] = False
    df['meu_9_3_condition'] = False
    df['meu_9_3_constraint'] = None

    # ===================== 1. 检查subject =====================
    # 标记valid的subject
    valid_subject = df['股东身份'].isin(['董监高'])
    df.loc[valid_subject, 'meu_9_3_subject'] = True

    # ===================== 2. 检查condition =====================
    # 计算公司上市时净利润
    ipo_profit = df.groupby('公司简称').apply(
        lambda x: x.set_index('日期').loc[x['上市日期'].iloc[0], '净利润']
    )
    df['_temp_ipo_profit'] = df['公司简称'].map(ipo_profit)
    
    # 检查上市至今是否曾盈利（上市后是否曾有净利润>0）
    since_ipo_profited = df.groupby('公司简称').apply(
        lambda x: x[x['日期'] >= x['上市日期'].iloc[0]]['净利润'].gt(0).any()
    )
    df['_temp_since_ipo_profited'] = df['公司简称'].map(since_ipo_profited)
    
    # 计算上市至今时长
    df['_temp_years_since_ipo'] = (df['日期'] - df['上市日期']).dt.days / 365.25
    
    # 检查是否有离任日期且在上市两年内
    df['_temp_has_left'] = df['离任日期'].notna()
    df['_temp_left_within_two_years'] = (df['离任日期'] - df['上市日期']).dt.days <= 730
    
    # condition条件：
    # 1. 公司上市时未盈利
    # 2. 公司上市至今未盈利
    # 3. 公司上市至今不满两年
    # 且在上市两年内离任
    df['meu_9_3_condition'] = (
        (df['_temp_ipo_profit'] <= 0) & 
        (~df['_temp_since_ipo_profited']) & 
        (df['_temp_years_since_ipo'] < 2) &
        df['_temp_has_left'] &
        df['_temp_left_within_two_years']
    )

    # ===================== 3. 检查constraint =====================
    # constraint保留None
    df['meu_9_3_constraint'] = None

    # 清理临时列
    df.drop(columns=[
        '_temp_ipo_profit', 
        '_temp_since_ipo_profited',
        '_temp_years_since_ipo',
        '_temp_has_left',
        '_temp_left_within_two_years'
    ], inplace=True)
    
    return df

In [ ]:
df = check_meu_9_3(df)
df

---

# Law Article 10

## MEU_10_1


| 字段 | 内容 |
|------|------|
| subject | 上市公司大股东 |
| condition | 因涉嫌与本上市公司有关的证券期货违法犯罪，在被中国证监会及其派出机构立案调查或者被司法机关立案侦查期间 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1711 |
| completion_tokens | 3253 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_10_1(df):
    '''
    验证MEU_10_1合规性：
    "subject": "上市公司大股东",
    "condition": "因涉嫌与本上市公司有关的证券期货违法犯罪，在被中国证监会及其派出机构立案调查或者被司法机关立案侦查期间",
    "constraint": "不得减持其所持有的本公司股份"
    
    改进点：处理调查开始但未结束的情况，标记直到最后记录日期
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_10_1_subject'] = False
    df['meu_10_1_condition'] = False
    df['meu_10_1_constraint'] = None

    # 1. 标记责任主体有效性（上市公司大股东）
    major_shareholder_mask = (
        df['股东身份'].isin(['控股股东', '实际控制人', '持股5%以上股东']) |
        (df['持股比例'] >= 0.05)
    )
    df.loc[major_shareholder_mask, 'meu_10_1_subject'] = True

    # 2. 标记触发条件有效性（处于被调查/侦查期间）
    investigation_events = [
        '被中国证监会及其派出机构立案调查',
        '中国证监会及其派出机构立案调查结束',
        '被司法机关立案侦查',
        '司法机关立案侦查结束'
    ]

    # 获取所有涉及调查/侦查的（公司简称 + 股东）组合
    involved_cases = df[df['股东涉嫌证券期货违法犯罪事件'].isin(investigation_events)][
        ['公司简称', '股东']
    ].drop_duplicates()

    # 获取每个(公司,股东)的最后记录日期
    last_dates = df.groupby(['公司简称', '股东'])['日期'].max().reset_index()
    
    for _, case in involved_cases.iterrows():
        company = case['公司简称']
        shareholder = case['股东']
        
        # 获取该（公司+股东）的所有事件并按时间排序
        case_events = df[
            (df['公司简称'] == company) & 
            (df['股东'] == shareholder)
        ][['日期', '股东涉嫌证券期货违法犯罪事件']].dropna().sort_values('日期')

        # 跟踪当前是否处于调查/侦查期间
        in_investigation = False
        investigation_start = None

        for _, row in case_events.iterrows():
            event = row['股东涉嫌证券期货违法犯罪事件']
            date = row['日期']

            if event == '被中国证监会及其派出机构立案调查':
                in_investigation = True
                investigation_start = date
            elif event == '中国证监会及其派出机构立案调查结束' and in_investigation:
                # 标记从立案到结案期间的所有记录
                df.loc[
                    (df['公司简称'] == company) & 
                    (df['股东'] == shareholder) & 
                    (df['日期'] >= investigation_start) & 
                    (df['日期'] <= date),
                    'meu_10_1_condition'
                ] = True
                in_investigation = False
            elif event == '被司法机关立案侦查':
                in_investigation = True
                investigation_start = date
            elif event == '司法机关立案侦查结束' and in_investigation:
                # 标记从立案到结案期间的所有记录
                df.loc[
                    (df['公司简称'] == company) & 
                    (df['股东'] == shareholder) & 
                    (df['日期'] >= investigation_start) & 
                    (df['日期'] <= date),
                    'meu_10_1_condition'
                ] = True
                in_investigation = False
        
        # 处理调查开始但未结束的情况
        if in_investigation and investigation_start:
            last_date = last_dates[
                (last_dates['公司简称'] == company) & 
                (last_dates['股东'] == shareholder)
            ]['日期'].values[0]
            
            # 标记从立案到最后记录日期的所有记录
            df.loc[
                (df['公司简称'] == company) & 
                (df['股东'] == shareholder) & 
                (df['日期'] >= investigation_start) & 
                (df['日期'] <= last_date),
                'meu_10_1_condition'
            ] = True

    # 3. 标记约束有效性（未减持股份）
    df['meu_10_1_constraint'] = df['当日减持比例'] <= 0

    return df

In [ ]:
df = check_meu_10_1(df)
df['meu_10_1_condition'].sum()

---

## MEU_10_2


| 字段 | 内容 |
|------|------|
| subject | 上市公司大股东 |
| condition | 因涉嫌与本上市公司有关的证券期货违法犯罪被中国证监会及其派出机构立案调查或者被司法机关立案侦查，在行政处罚决定、刑事判决作出之后未满6个月 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1722 |
| completion_tokens | 8204 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_10_2(df):
    '''
    验证MEU_10_2合规性：
    - subject: 上市公司大股东（控股股东/实际控制人/持股5%以上股东）
    - condition: 因证券违法被处罚/判决后6个月内
    - constraint: 不得减持股份
    '''
    df = df.copy()

    # 初始化合规标记列
    df['meu_10_2_subject'] = False
    df['meu_10_2_condition'] = False
    df['meu_10_2_constraint'] = None

    # 1. 验证责任主体（上市公司大股东）
    subject_mask = (
        df['股东身份'].isin(['控股股东', '实际控制人', '持股5%以上股东']) | 
        (df['持股比例'] >= 0.05)
    )
    df.loc[subject_mask, 'meu_10_2_subject'] = True

    # 2. 验证触发条件（只考虑处罚/判决后180天内）
    # 预处理处罚/判决时间点
    penalty_events = ['行政处罚决定作出', '刑事判决作出']
    penalty_dates = df[df['股东涉嫌证券期货违法犯罪事件'].isin(penalty_events)]
    penalty_dates = penalty_dates.groupby(['公司简称', '股东'])['日期'].apply(list).reset_index()

    # 合并特征数据
    df = df.merge(penalty_dates.rename(columns={'日期': 'penalty_dates'}),
                 on=['公司简称', '股东'], how='left')
    
    # 处理空值
    df['penalty_dates'] = df['penalty_dates'].apply(lambda x: x if isinstance(x, list) else [])

    # 条件验证函数
    def check_condition(row):
        for penalty_date in row['penalty_dates']:
            if (row['日期'] - penalty_date).days >= 0 and \
               (row['日期'] - penalty_date).days <= 180:
                return True
        return False

    df['meu_10_2_condition'] = df.apply(check_condition, axis=1)

    # 3. 验证约束条件（独立检查）
    df['meu_10_2_constraint'] = df['当日减持比例'] == 0

    # 清理中间列
    df.drop(columns=['penalty_dates'], inplace=True, errors='ignore')

    return df

In [ ]:
df = check_meu_10_2(df)
df['meu_10_2_condition'].sum()

In [ ]:
df[~df['股东涉嫌证券期货违法犯罪事件'].isna()]['股东涉嫌证券期货违法犯罪事件']

In [ ]:
df[['日期', '股东涉嫌证券期货违法犯罪事件','meu_10_2_condition']][56230:56351]

In [ ]:
df[df['meu_10_2_condition']==True]

---

## MEU_10_3


| 字段 | 内容 |
|------|------|
| subject | 上市公司大股东 |
| condition | 因涉及与本上市公司有关的违法违规，被本所公开谴责未满3个月 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1704 |
| completion_tokens | 5748 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_10_3(df):
    '''
    检查MEU_10_3合规性：
    subject: 上市公司大股东
    condition: 因涉及违法违规被本所公开谴责未满3个月
    constraint: 不得减持股份
    '''
    df = df.copy()
    
    # 初始化合规标记列
    df['meu_10_3_subject'] = False
    df['meu_10_3_condition'] = False
    df['meu_10_3_constraint'] = None
    
    # 1. 验证责任主体
    is_major_shareholder = (
        df['股东身份'].isin(['控股股东', '实际控制人', '持股5%以上股东']) |
        (df['持股比例'] >= 0.05)
    )
    df.loc[is_major_shareholder, 'meu_10_3_subject'] = True
    
    # 2. 验证触发条件（只考虑处罚/判决后180天内）
    # 预处理处罚/判决时间点
    penalty_events = ['被本所公开谴责']
    penalty_dates = df[df['股东涉嫌证券期货违法犯罪事件'].isin(penalty_events)]
    penalty_dates = penalty_dates.groupby(['公司简称', '股东'])['日期'].apply(list).reset_index()

    # 合并特征数据
    df = df.merge(penalty_dates.rename(columns={'日期': 'penalty_dates'}),
                 on=['公司简称', '股东'], how='left')
    
    # 处理空值
    df['penalty_dates'] = df['penalty_dates'].apply(lambda x: x if isinstance(x, list) else [])

    # 条件验证函数
    def check_condition(row):
        for penalty_date in row['penalty_dates']:
            if (row['日期'] - penalty_date).days >= 0 and \
               (row['日期'] - penalty_date).days <= 180:
                return True
        return False

    df['meu_10_3_condition'] = df.apply(check_condition, axis=1)
    
    # 3. 验证约束条件（保持原有逻辑）
    df['meu_10_3_constraint'] = df['当日减持比例'] <= 0
    
    return df

In [ ]:
df = check_meu_10_3(df)
df['meu_10_3_condition'].sum()

In [ ]:
df[['日期', '股东', '股东涉嫌证券期货违法犯罪事件', 'meu_10_3_condition']][75:195]

In [ ]:
from datetime import datetime

# 定义两个日期
date1 = datetime.strptime("2021-02-18", "%Y-%m-%d")
date2 = datetime.strptime("2020-08-21", "%Y-%m-%d")

# 计算日期差（绝对值）
delta = abs(date1 - date2)

# 输出自然日天数
print(f"2021-02-18到2020-08-21之间有{delta.days}个自然日")

In [ ]:
df[df['meu_10_3_condition']==True][['日期', '股东', '股东涉嫌证券期货违法犯罪事件']]

---

## MEU_10_4


| 字段 | 内容 |
|------|------|
| subject | 上市公司大股东 |
| condition | 因涉及证券期货违法，被中国证监会行政处罚且罚没款尚未足额缴纳，且不存在法律、行政法规另有规定或减持资金用于缴纳罚没款的情形 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1723 |
| completion_tokens | 7540 |


### 代码实现

In [ ]:
# # 读取模拟数据
# import pandas as pd
# df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
# date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
# for col in date_columns:
#     df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
# import pandas as pd

# def check_meu_10_4(df):
#     '''
#     验证MEU_10_4合规性：
#     subject: 上市公司大股东
#     condition: 因证券期货违法被行政处罚且罚没款未缴，且不存在例外情形
#     constraint: 不得减持股份
#     '''
#     df = df.copy()

#     # 初始化合规性标记列
#     df['meu_10_4_subject'] = False
#     df['meu_10_4_condition'] = False
#     df['meu_10_4_constraint'] = None

#     # 1. 验证责任主体（上市公司大股东）
#     major_shareholder_mask = (
#         df['股东身份'].isin(['控股股东', '实际控制人', '持股5%以上股东']) | 
#         (df['持股比例'] >= 0.05)
#     )
#     df.loc[major_shareholder_mask, 'meu_10_4_subject'] = True

#     # 2. 验证触发条件（行政处罚+未足额缴纳+无例外情形）
#     # 按时间顺序计算行政处罚持续状态
#     sorted_df = df.sort_values(['公司简称', '股东', '日期'])
#     sorted_df['行政处罚持续状态'] = (
#         (sorted_df['股东涉嫌证券期货违法犯罪事件'] == '行政处罚决定作出')
#         .groupby([sorted_df['公司简称'], sorted_df['股东']])
#         .cummax()
#     )
#     df['行政处罚持续状态'] = sorted_df['行政处罚持续状态'].values  # 保持原始索引顺序
    
#     # 检查例外情形（减持资金用于缴纳罚没款）
#     exception_mask = df['拟减持原因'] == '缴纳罚没款'
#     valid_condition = df['行政处罚持续状态'] & ~exception_mask
#     df.loc[valid_condition, 'meu_10_4_condition'] = True

#     # 3. 验证约束条件（当日无减持）
#     df['meu_10_4_constraint'] = df['当日减持比例'].eq(0)

#     # 清理临时字段
#     df.drop('行政处罚持续状态', axis=1, inplace=True)
    
#     return df

In [ ]:
# df = check_meu_10_4(df)
# df

---

# Law Article 11

## MEU_11_1


| 字段 | 内容 |
|------|------|
| subject | 上市公司控股股东 \| 实际控制人 |
| condition | 上市公司因涉嫌证券期货违法犯罪，在被中国证监会及其派出机构立案调查或者被司法机关立案侦查期间 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1714 |
| completion_tokens | 4512 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_11_1(df):
    '''
    验证MEU_11_1合规性：
    "subject": "上市公司控股股东, 实际控制人",
    "condition": "因涉嫌与本上市公司有关的证券期货违法犯罪，在被中国证监会及其派出机构立案调查或者被司法机关立案侦查期间",
    "constraint": "不得减持其所持有的本公司股份"
    
    改进点：处理调查开始但未结束的情况，标记直到最后记录日期
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_11_1_subject'] = False
    df['meu_11_1_condition'] = False
    df['meu_11_1_constraint'] = None

    # 1. 标记责任主体有效性（控股股东）
    major_shareholder_mask = (
        df['股东身份'].isin(['控股股东', '实际控制人'])
    )
    df.loc[major_shareholder_mask, 'meu_11_1_subject'] = True

    # 2. 标记触发条件有效性（处于被调查/侦查期间）
    investigation_events = [
        '被中国证监会及其派出机构立案调查',
        '中国证监会及其派出机构立案调查结束',
        '被司法机关立案侦查',
        '司法机关立案侦查结束'
    ]

    # 获取所有涉及调查/侦查的（公司简称 + 股东）组合
    involved_cases = df[df['公司涉嫌证券期货违法犯罪事件'].isin(investigation_events)][
        ['公司简称', '股东']
    ].drop_duplicates()

    # 获取每个(公司,股东)的最后记录日期
    last_dates = df.groupby(['公司简称', '股东'])['日期'].max().reset_index()
    
    for _, case in involved_cases.iterrows():
        company = case['公司简称']
        shareholder = case['股东']
        
        # 获取该（公司+股东）的所有事件并按时间排序
        case_events = df[
            (df['公司简称'] == company) & 
            (df['股东'] == shareholder)
        ][['日期', '公司涉嫌证券期货违法犯罪事件']].dropna().sort_values('日期')

        # 跟踪当前是否处于调查/侦查期间
        in_investigation = False
        investigation_start = None

        for _, row in case_events.iterrows():
            event = row['公司涉嫌证券期货违法犯罪事件']
            date = row['日期']

            if event == '被中国证监会及其派出机构立案调查':
                in_investigation = True
                investigation_start = date
            elif event == '中国证监会及其派出机构立案调查结束' and in_investigation:
                # 标记从立案到结案期间的所有记录
                df.loc[
                    (df['公司简称'] == company) & 
                    (df['股东'] == shareholder) & 
                    (df['日期'] >= investigation_start) & 
                    (df['日期'] <= date),
                    'meu_11_1_condition'
                ] = True
                in_investigation = False
            elif event == '被司法机关立案侦查':
                in_investigation = True
                investigation_start = date
            elif event == '司法机关立案侦查结束' and in_investigation :
                # 标记从立案到结案期间的所有记录
                df.loc[
                    (df['公司简称'] == company) & 
                    (df['股东'] == shareholder) & 
                    (df['日期'] >= investigation_start) & 
                    (df['日期'] <= date),
                    'meu_11_1_condition'
                ] = True
                in_investigation = False
        
        # 处理调查开始但未结束的情况
        if in_investigation and investigation_start:
            last_date = last_dates[
                (last_dates['公司简称'] == company) & 
                (last_dates['股东'] == shareholder)
            ]['日期'].values[0]
            
            # 标记从立案到最后记录日期的所有记录
            df.loc[
                (df['公司简称'] == company) & 
                (df['股东'] == shareholder) & 
                (df['日期'] >= investigation_start) & 
                (df['日期'] <= last_date),
                'meu_11_1_condition'
            ] = True

    # 3. 标记约束有效性（未减持股份）
    df['meu_11_1_constraint'] = df['当日减持比例'] <= 0

    return df


In [ ]:
df = check_meu_11_1(df)
df['meu_11_1_condition'].sum()

---

## MEU_11_2


| 字段 | 内容 |
|------|------|
| subject | 上市公司控股股东 \| 实际控制人 |
| condition | 上市公司因涉嫌证券期货违法犯罪被中国证监会及其派出机构立案调查或者被司法机关立案侦查，在行政处罚决定、刑事判决作出之后未满6个月的 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1725 |
| completion_tokens | 5506 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_11_2(df):
    '''
    验证MEU_11_2合规性：
    subject: 上市公司控股股东 | 实际控制人
    condition: 公司被行政处罚/刑事判决后6个月内(180个自然日)
    constraint: 不得减持股份
    
    改进点：统一时间范围处理逻辑，确保处罚后180天内都被标记
    '''
    df = df.copy()

    # 初始化合规标记列
    df['meu_11_2_subject'] = False
    df['meu_11_2_condition'] = False
    df['meu_11_2_constraint'] = None

    # 1. 验证责任主体（控股股东或实际控制人）
    major_shareholder_mask = df['股东身份'].isin(['控股股东', '实际控制人'])
    df.loc[major_shareholder_mask, 'meu_11_2_subject'] = True

    # 2. 验证触发条件（处罚后180天内）
    penalty_events = ['行政处罚决定作出', '刑事判决作出']
    
    # 获取所有涉及处罚的公司
    penalized_companies = df[df['公司涉嫌证券期货违法犯罪事件'].isin(penalty_events)][
        ['公司简称']
    ].drop_duplicates()
    
    # 获取每个公司的最后处罚日期
    last_penalty_dates = df[df['公司涉嫌证券期货违法犯罪事件'].isin(penalty_events)]\
        .groupby('公司简称')['日期'].max().reset_index()
    
    for _, company in penalized_companies.iterrows():
        company_name = company['公司简称']
        
        # 获取该公司的最新处罚日期
        penalty_date = last_penalty_dates[
            last_penalty_dates['公司简称'] == company_name
        ]['日期'].values[0]
        
        # 计算180天后的日期
        penalty_end_date = penalty_date + pd.Timedelta(days=180)
        
        # 获取该公司的最后记录日期（用于处理未满180天的情况）
        last_record_date = df[df['公司简称'] == company_name]['日期'].max()
        
        # 实际结束日期取两者中较早的（180天或最后记录日期）
        effective_end_date = min(penalty_end_date, last_record_date)
        
        # 标记处罚后180天内（或直到最后记录日期）的所有记录
        df.loc[
            (df['公司简称'] == company_name) & 
            (df['日期'] >= penalty_date) & 
            (df['日期'] <= effective_end_date),
            'meu_11_2_condition'
        ] = True

    # 3. 验证约束条件（未减持股份）
    df['meu_11_2_constraint'] = df['当日减持比例'] <= 0

    return df

In [ ]:
df = check_meu_11_2(df)
df

---

## MEU_11_3


| 字段 | 内容 |
|------|------|
| subject | 上市公司控股股东 | 实际控制人 |
| condition | 上市公司被本所公开谴责未满3个月 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1703 |
| completion_tokens | 5146 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_11_3(df):
    '''
    验证MEU_11_3合规性（完整180天窗口版）：
    "subject": "上市公司控股股东 | 实际控制人", 
    "condition": "上市公司被本所公开谴责未满6个月(180个自然日)", 
    "constraint": "不得减持其所持有的本公司股份"
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_11_3_subject'] = False
    df['meu_11_3_condition'] = False
    df['meu_11_3_constraint'] = None

    # 1. 标记责任主体（控股股东或实际控制人）
    df['meu_11_3_subject'] = df['股东身份'].isin(['控股股东', '实际控制人'])

    # 2. 处理公开谴责时间窗口
    condemn_events = df[df['公司涉嫌证券期货违法犯罪事件'] == '被本所公开谴责']
    
    if not condemn_events.empty:
        # 正确使用agg方法获取谴责日期和窗口结束日期
        condemn_windows = condemn_events.groupby('公司简称')['日期'].agg([
            ('condemn_date', 'min'),  # 最早谴责日期
            ('window_end', lambda x: x.min() + pd.Timedelta(days=180))  # 180天后
        ]).reset_index()
        
        # 为每个公司创建时间窗口标记
        for _, row in condemn_windows.iterrows():
            window_mask = (
                (df['公司简称'] == row['公司简称']) &
                (df['日期'] >= row['condemn_date']) &
                (df['日期'] <= row['window_end'])
            )
            df.loc[window_mask, 'meu_11_3_condition'] = True

    # 3. 处理减持约束（NaN视为0）
    df['当日减持比例'] = df['当日减持比例'].fillna(0)
    df['meu_11_3_constraint'] = (df['当日减持比例'] <= 0)

    return df

In [ ]:
df = check_meu_11_3(df)
df

In [ ]:
df[df['meu_11_3_condition']==True][['日期']]

In [ ]:
df[['日期', '公司简称', '公司涉嫌证券期货违法犯罪事件','meu_11_3_condition']][520:646]

---

## MEU_11_4


| 字段 | 内容 |
|------|------|
| subject | 上市公司控股股东 | 实际控制人 |
| condition | 市公司股票因可能触及重大违法强制退市情形，而被本所实施退市风险警示，在本所规定的限制减持期限内的 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1722 |
| completion_tokens | 7246 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_11_4(df):
    '''
    检查MEU_11_4合规性：
    subject: 控股股东或实际控制人
    condition: 公司因重大违法被实施退市风险警示且在限制期内
    constraint: 不得减持股份
    
    改进点：处理"退市风险警示"列中的'限制减持期限开始'和'限制减持期限结束'事件，
            标记这两个日期之间的记录为condition有效
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_11_4_subject'] = False
    df['meu_11_4_condition'] = False
    df['meu_11_4_constraint'] = None

    # 1. 标记责任主体有效性（控股股东或实际控制人）
    valid_subject = df['股东身份'].isin(['控股股东', '实际控制人'])
    df.loc[valid_subject, 'meu_11_4_subject'] = True

    # 2. 标记触发条件有效性（处于限制减持期限内）
    restriction_events = [
        '限制减持期限开始',
        '限制减持期限结束'
    ]

    # 获取所有涉及限制减持的（公司简称 + 股东）组合
    involved_cases = df[df['退市风险警示'].isin(restriction_events)][
        ['公司简称', '股东']
    ].drop_duplicates()

    # 获取每个(公司,股东)的最后记录日期
    last_dates = df.groupby(['公司简称', '股东'])['日期'].max().reset_index()
    
    for _, case in involved_cases.iterrows():
        company = case['公司简称']
        shareholder = case['股东']
        
        # 获取该（公司+股东）的所有退市风险警示事件并按时间排序
        case_events = df[
            (df['公司简称'] == company) & 
            (df['股东'] == shareholder) & 
            (df['退市风险警示'].isin(restriction_events))
        ][['日期', '退市风险警示']].dropna().sort_values('日期')

        # 跟踪当前是否处于限制减持期间
        in_restriction = False
        restriction_start = None

        for _, row in case_events.iterrows():
            event = row['退市风险警示']
            date = row['日期']

            if event == '限制减持期限开始':
                in_restriction = True
                restriction_start = date
            elif event == '限制减持期限结束' and in_restriction:
                # 标记从开始到结束期间的所有记录
                df.loc[
                    (df['公司简称'] == company) & 
                    (df['股东'] == shareholder) & 
                    (df['日期'] >= restriction_start) & 
                    (df['日期'] <= date),
                    'meu_11_4_condition'
                ] = True
                in_restriction = False
        
        # 处理限制期开始但未结束的情况
        if in_restriction and restriction_start:
            last_date = last_dates[
                (last_dates['公司简称'] == company) & 
                (last_dates['股东'] == shareholder)
            ]['日期'].values[0]
            
            # 标记从开始到最后记录日期的所有记录
            df.loc[
                (df['公司简称'] == company) & 
                (df['股东'] == shareholder) & 
                (df['日期'] >= restriction_start) & 
                (df['日期'] <= last_date),
                'meu_11_4_condition'
            ] = True

    # 3. 标记约束有效性（未减持股份）
    df['meu_11_4_constraint'] = (df['当日减持比例'] == 0)

    return df

In [ ]:
df = check_meu_11_4(df)
df

In [ ]:
df[df['meu_11_4_condition']==True]

In [ ]:
df[['日期', '股东', '退市风险警示', 'meu_11_4_condition']][179:247]

---

# Law Article 12

## MEU_12_1


| 字段 | 内容 |
|------|------|
| subject | 公开发行股票并上市时的控股股东 \| 公开发行股票并上市时的实际控制人 |
| condition | 计划通过集中竞价交易或大宗交易减持股份且首次披露减持计划, 且不存在已经按照本指引第四条披露减持计划，或者中国证监会另有规定的情况的 |
| constrain | 不得存在下列情形：最近20个交易日内任一交易日股票收盘价低于公开发行股票并上市的发行价格 |
| contextual_info | 股票收盘价以公开发行股票并上市之日为基准向后复权计算 |
| note | 不考虑中国证监会另有规定的情况 |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1778 |
| completion_tokens | 6948 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
def check_meu_12_1(df):
    '''
    检查MEU_12_1合规性：
    subject: 公开发行股票并上市时的控股股东 | 实际控制人
    condition: 计划通过集中竞价或大宗交易减持股份且首次披露减持计划
    constraint: 最近20个交易日内任一交易日复权收盘价不低于发行价
    contextual_info: 复权计算以发行日为准
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_12_1_subject'] = False
    df['meu_12_1_condition'] = False
    df['meu_12_1_constraint'] = None

    # 处理subject部分
    # 构建公司上市时控股股东/实际控制人字典
    ipo_entities = (df.sort_values('日期')
                    .groupby('公司简称')
                    .apply(lambda g: set(g[g['日期'] == g['上市日期'].iloc[0]]
                                        .query("股东身份 in ['控股股东','实际控制人']")['股东'])))
    
    # 标记符合主体条件
    df['is_ipo_entity'] = df.apply(
        lambda x: x['股东'] in ipo_entities.get(x['公司简称'], set()), axis=1)
    valid_subject = df['is_ipo_entity'] 
    df.loc[valid_subject, 'meu_12_1_subject'] = True

    # 处理condition部分
    # 获取首次减持计划披露日
    has_plan = df[df['存在减持计划']]
    first_disclosure = has_plan.groupby(['公司简称', '股东'])['日期'].min().reset_index(name='首次披露日')
    
    # 合并首次披露日并验证条件
    df = df.merge(first_disclosure, on=['公司简称', '股东'], how='left')
    valid_condition = (
        df['减持方式'].isin(['竞价交易', '大宗交易']) & 
        df['存在减持计划'] & 
        (df['日期'] == df['首次披露日'])
    )
    df['meu_12_1_condition'] = valid_condition

    # 处理constraint部分
    # 计算复权价格并标记价格违规
    # 复权因子是以上市日为基准的, 所以可以直接计算
    df['复权收盘价'] = df['原始收盘价'] * df['复权因子']
    df['价格违规'] = df['复权收盘价'] < df['发行价格']
    
    # 按公司滚动检查20日窗口
    def check_price_violation(group):
        group = group.sort_values('日期')
        # 使用max()替代apply(any)以确保布尔类型输出
        group['20日违规标记'] = group['价格违规'].rolling(20, min_periods=1).max().astype(bool)
        return group
    
    df = df.groupby('公司简称', group_keys=False).apply(check_price_violation)
    df['meu_12_1_constraint'] = ~df['20日违规标记']

    # 清理中间列
    df.drop([
        '公司上市日期', '首次披露日', 
        '复权收盘价', '价格违规', '20日违规标记'
    ], axis=1, errors='ignore', inplace=True)

    return df

In [ ]:
df = check_meu_12_1(df)
df[(df['is_ipo_entity']==True) & (df['日期']==df['上市日期'])][['日期', '股东', '股东身份']]

In [ ]:
unique_shareholders = df[(df['is_ipo_entity']==True)][['日期', '股东', '股东身份']]['股东'].unique()
unique_shareholders

---

## MEU_12_2


| 字段 | 内容 |
|------|------|
| subject | 上市公司控股股东 \| 实际控制人 \| 一致行动人 |
| condition | 计划通过集中竞价交易或大宗交易减持股份且首次披露减持计划, 且不存在已经按照本指引第四条披露减持计划，或者中国证监会另有规定的情况的 |
| constrain | 不得存在下列情形：最近20个交易日内，上市公司任一交易日股票收盘价低于最近一个会计年度或者最近一期财务会计报告期末每股归属于上市公司股东的净资产 |
| contextual_info | 股票收盘价以最近一个会计年度或者最近一期财务会计报告资产负债表日为基准分别向后复权计算 |
| note | 不考虑中国证监会另有规定的情况 |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1787 |
| completion_tokens | 6694 |


### 代码实现

In [23]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期', '资产负债表日',]
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_12_2(df):
    '''
    检查MEU_12_1的合规性：
    主体：控股股东 | 实际控制人
    条件：首次通过集中竞价或大宗交易减持计划
    约束：最近20个交易日内股价不低于最近一期每股净资产（后复权）
    '''
    df = df.copy()
    
    # 初始化标记列
    df['meu_12_2_subject'] = False
    df['meu_12_2_condition'] = False
    df['meu_12_2_constraint'] = None

    # 1. 标记责任主体有效性（控股股东或实际控制人）
    valid_subject = df['股东身份'].isin(['控股股东', '实际控制人'])
    df.loc[valid_subject, 'meu_12_2_subject'] = True

    # 处理condition部分
    # 获取首次减持计划披露日
    has_plan = df[df['存在减持计划']]
    first_disclosure = has_plan.groupby(['公司简称', '股东'])['日期'].min().reset_index(name='首次披露日')
    
    # 合并首次披露日并验证条件
    df = df.merge(first_disclosure, on=['公司简称', '股东'], how='left')
    valid_condition = (
        df['减持方式'].isin(['竞价交易', '大宗交易']) & 
        df['存在减持计划'] & 
        (df['日期'] == df['首次披露日'])
    )
    df['meu_12_2_condition'] = valid_condition
    
    # 3. 验证约束条件（最近20个交易日股价不低于每股净资产）
    # 预处理：为每个公司建立复权因子和收盘价的时间序列
    company_data = {}
    for company in df['公司简称'].unique():
        # 按日期排序，保留最新的复权因子（假设每天有最新值）
        company_df = df[df['公司简称'] == company][['日期', '原始收盘价', '复权因子']].drop_duplicates('日期', keep='last').sort_values('日期')
        company_data[company] = company_df.set_index('日期')

    def get_adjusted_prices(company, base_date, window_dates):
        """获取基于基准日期的复权收盘价"""
        cf = company_data.get(company)

        # 获取基准复权因子（财务数据日期当天的复权因子）
        base_factor = cf['复权因子'].asof(base_date)


        # 合并窗口日期与公司数据
        window_df = pd.DataFrame({'日期': window_dates})
        merged = pd.merge_asof(window_df.sort_values('日期'),
                              cf.reset_index().sort_values('日期'),
                              on='日期',
                              direction='backward')
        
        # 计算复权价格
        merged['adjusted'] = merged['原始收盘价'] * (merged['复权因子'] / base_factor)
        print(base_factor)
        print(merged['adjusted'])
        return merged.set_index('日期')['adjusted']
    
    
    # 构建公司交易日历
    company_calendars = df.groupby('公司简称')['日期'].unique().apply(sorted).to_dict()

    # 处理约束条件
    mask = df['meu_12_2_subject'] & df['meu_12_2_condition']
    for idx in df[mask].index:
        row = df.loc[idx]
        company = row['公司简称']
        current_date = row['日期']
        base_date = row['资产负债表日'] 
        net_per_share = row['每股净资产']

        # 获取交易日历
        all_dates = company_calendars.get(company, [])
        pos = all_dates.index(current_date)
        
            
        # 取最近20个交易日
        window_dates = all_dates[max(0, pos-19):pos+1]

        # 获取复权收盘价
        adjusted_prices = get_adjusted_prices(company, base_date, window_dates)
        
        # 检查约束
        if not adjusted_prices.empty:
            violation = (adjusted_prices < net_per_share).any()
            df.loc[idx, 'meu_12_2_constraint'] = not violation

    # 清理中间列
    df.drop(['is_ipo_entity', '首次披露日'], axis=1, errors='ignore', inplace=True)
    
    return df


In [25]:
df = check_meu_12_2(df)
df

/var/folders/0l/pjlf4pl90rx7ldkq7jx18ny00000gn/T/ipykernel_3744/929462158.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: set(g[g['日期'] == g['上市日期'].iloc[0]]


nan
0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
5    NaN
6    NaN
7    NaN
8    NaN
9    NaN
10   NaN
11   NaN
12   NaN
13   NaN
14   NaN
15   NaN
16   NaN
17   NaN
18   NaN
19   NaN
Name: adjusted, dtype: float64
nan
0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
5    NaN
6    NaN
7    NaN
8    NaN
9    NaN
10   NaN
11   NaN
12   NaN
13   NaN
14   NaN
15   NaN
16   NaN
17   NaN
18   NaN
19   NaN
Name: adjusted, dtype: float64
1.0
0     14.525149
1     14.196971
2     14.561115
3     14.142183
4     14.466822
5     14.315407
6     13.969252
7     14.089220
8     14.093432
9     14.150497
10    14.280015
11    14.417633
12    13.852602
13    13.402560
14    12.959960
15    12.922803
16    12.863842
17    12.693577
18    12.714980
19    13.061730
Name: adjusted, dtype: float64


,日期,日收益率,收盘价,前收盘价,上市日期,发行价格,公司行为,复权因子,总股本,原始收盘价,...,拟减持原因,持股数量,当日减持数量,股份来源,公司简称,股东,股东涉嫌证券期货违法犯罪事件,meu_12_2_subject,meu_12_2_condition,meu_12_2_constraint
0,2020-04-30,NaN,NaN,NaN,2021-05-18,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,北交所上市前取得,钧璋机械,马翊璟,NaN,False,False,None
1,2020-05-06,NaN,NaN,NaN,2021-05-18,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,北交所上市前取得,钧璋机械,马翊璟,NaN,False,False,None
2,2020-05-07,NaN,NaN,NaN,2021-05-18,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,北交所上市前取得,钧璋机械,马翊璟,NaN,False,False,None
3,2020-05-08,NaN,NaN,NaN,2021-05-18,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,北交所上市前取得,钧璋机械,马翊璟,NaN,False,False,None
4,2020-05-11,NaN,NaN,NaN,2021-05-18,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,北交所上市前取得,钧璋机械,马翊璟,NaN,False,False,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56745,2024-12-25,0.006500,41.514442,41.246344,2021-06-23,34.538724,NaN,1.166629,1.142639e+08,35.584964,...,NaN,8.184114e+06,0.0,北交所上市前取得,沅璐股份,玥滢控股,NaN,False,False,None
56746,2024-12-26,0.021031,42.387518,41.514442,2021-06-23,34.538724,NaN,1.166629,1.142639e+08,36.333339,...,NaN,8.184114e+06,0.0,北交所上市前取得,沅璐股份,玥滢控股,NaN,False,False,None
56747,2024-12-27,0.011522,42.875886,42.387518,2021-06-23,34.538724,NaN,1.166629,1.142639e+08,36.751954,...,NaN,8.184114e+06,0.0,北交所上市前取得,沅璐股份,玥滢控股,NaN,False,False,None
56748,2024-12-30,-0.006458,42.599006,42.875886,2021-06-23,34.538724,NaN,1.166629,1.142639e+08,36.514621,...,NaN,8.184114e+06,0.0,北交所上市前取得,沅璐股份,玥滢控股,NaN,False,False,None


In [21]:
df.to_csv('temp_meu_18_2.csv', encoding='utf-8-sig', index=False)

---

## MEU_12_3


| 字段 | 内容 |
|------|------|
| subject | 上市公司控股股东 \| 实际控制人 \| 一致行动人 |
| condition | 计划通过集中竞价交易或大宗交易减持股份且首次披露减持计划, 且不存在已经按照本指引第四条披露减持计划，或者中国证监会另有规定的情况的 |
| constrain | 不得存在下列情形：上市公司最近一期经审计的财务报告的归属于上市公司股东的净利润为负 |
| contextual_info | nan |
| note | 不考虑中国证监会另有规定的情况 |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1752 |
| completion_tokens | 3516 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_12_3(df):
    '''
    检查MEU_12_3合规性：
    subject: 上市公司控股股东 | 实际控制人
    condition: 计划通过集中竞价或大宗交易减持股份且首次披露减持计划
    constraint: 最近一期经审计净利润不得为负
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_12_3_subject'] = False
    df['meu_12_3_condition'] = False
    df['meu_12_3_constraint'] = None

    # 1. 标记valid的subject
    # 处理控股股东和实际控制人（数据中无一致行动人字段）
    valid_subject = df['股东身份'].isin(['控股股东', '实际控制人'])
    df.loc[valid_subject, 'meu_12_3_subject'] = True

    # 处理condition部分
    # 获取首次减持计划披露日
    has_plan = df[df['存在减持计划']]
    first_disclosure = has_plan.groupby(['公司简称', '股东'])['日期'].min().reset_index(name='首次披露日')
    # 合并首次披露日并验证条件
    df = df.merge(first_disclosure, on=['公司简称', '股东'], how='left')
    valid_condition = (
        df['减持方式'].isin(['竞价交易', '大宗交易']) & 
        df['存在减持计划'] & 
        (df['日期'] == df['首次披露日'])
    )
    df['meu_12_3_condition'] = valid_condition

    # 3. 标记valid的constraint（独立检查）
    # 净利润>=0时满足约束
    df['meu_12_3_constraint'] = df['净利润'] >= 0

    # 清理临时列
    df.drop('prev_plan_count', axis=1, inplace=True, errors='ignore')

    return df

In [ ]:
df = check_meu_12_3(df)
df[(df['公司简称'] == '鸾璐科技') & (df['存在减持计划'] == True)][['日期', '存在减持计划', '首次披露日']]

---

## MEU_12_4


| 字段 | 内容 |
|------|------|
| subject | 控股股东 \| 实际控制人（上市后不再具备相关主体身份的仍然遵守） |
| condition | 计划通过集中竞价交易或大宗交易减持股份且首次披露减持计划, 且不存在已经按照本指引第四条披露减持计划，或者中国证监会另有规定的情况的 |
| constrain | 不得存在下列情形：最近20个交易日内任一交易日股票收盘价低于公开发行股票并上市的发行价格 |
| contextual_info | 股票收盘价以公开发行股票并上市之日为基准向后复权计算 |
| note | 不考虑中国证监会另有规定的情况 |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1775 |
| completion_tokens | 7249 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_12_4(df):
    '''
    验证MEU_12_4合规性：
    subject: 上市时是控股股东/实际控制人 (当前已不具备该身份的也应当遵守)
    condition: 首次披露竞价/大宗减持计划
    constraint: 最近20交易日无复权收盘价低于发行价
    '''
    df = df.copy()
    
    # 初始化结果列
    df['meu_12_4_subject'] = False
    df['meu_12_4_condition'] = False
    df['meu_12_4_constraint'] = None
    
    # ================== SUBJECT验证 ==================
    # 构建公司上市时控股股东/实际控制人字典
    ipo_entities = (df.sort_values('日期')
                    .groupby('公司简称')
                    .apply(lambda g: set(g[g['日期'] == g['上市日期'].iloc[0]]
                                        .query("股东身份 in ['控股股东','实际控制人']")['股东'])))
    
    # 标记符合主体条件
    df['is_ipo_entity'] = df.apply(
        lambda x: x['股东'] in ipo_entities.get(x['公司简称'], set()), axis=1)
    valid_subject = df['is_ipo_entity'] 
    df.loc[valid_subject, 'meu_12_4_subject'] = True

    # ================== CONDITION验证 ==================
    # 处理condition部分
    # 获取首次减持计划披露日
    has_plan = df[df['存在减持计划']]
    first_disclosure = has_plan.groupby(['公司简称', '股东'])['日期'].min().reset_index(name='首次披露日')
    # 合并首次披露日并验证条件
    df = df.merge(first_disclosure, on=['公司简称', '股东'], how='left')
    valid_condition = (
        df['减持方式'].isin(['竞价交易', '大宗交易']) & 
        df['存在减持计划'] & 
        (df['日期'] == df['首次披露日'])
    )
    df['meu_12_4_condition'] = valid_condition

    # ================== CONSTRAINT验证 ==================
    # 计算复权价格
    df['复权价'] = df['原始收盘价'] * df['复权因子']
    
    # 按公司计算滚动窗口内最低价
    df_sorted = df.sort_values(['公司简称','日期'])
    df_sorted['发行价'] = df_sorted.groupby('公司简称')['发行价格'].transform('first')
    window_check = df_sorted.groupby('公司简称', group_keys=False).apply(
        lambda g: g['复权价'].rolling(20, min_periods=1).apply(
            lambda x: (x < g['发行价'].iloc[0]).any()))
    
    # 标记约束违规
    df_sorted['meu_12_4_constraint'] = ~(window_check > 0)
    df = df_sorted.sort_index().drop(columns=['复权价','发行价'])
    
    return df.drop(columns=['is_ipo_entity','首披日'], errors='ignore')


In [ ]:
df = check_meu_12_4(df)
df

In [ ]:
df['发行价格']

---

## MEU_12_5


| 字段 | 内容 |
|------|------|
| subject | 公开发行时持股5%以上的第一大股东 | 一致行动人 |
| condition | 上市公司在公开发行股票并上市时披露为无控股股东、实际控制人, 且计划通过集中竞价交易或大宗交易减持股份且首次披露减持计划, 且不存在已经按照本指引第四条披露减持计划，或者中国证监会另有规定的情况的 |
| constrain | 不得存在下列情形：最近20个交易日内任一交易日股票收盘价低于公开发行股票并上市的发行价格 |
| contextual_info | 股票收盘价以公开发行股票并上市之日为基准向后复权计算 |
| note | 不考虑中国证监会另有规定的情况 |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1793 |
| completion_tokens | 6905 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_12_5(df):
    """ 
    subject: 公开发行时持股5%以上的第一大股东
    condition: 上市公司在公开发行股票并上市时披露为无控股股东、实际控制人, 且计划通过集中竞价交易或大宗交易减持股份且首次披露减持计划, 且不存在已经按照本指引第四条披露减持计划，或者中国证监会另有规定的情况的
    constrain: 不得存在下列情形：最近20个交易日内任一交易日股票收盘价低于公开发行股票并上市的发行价格
    contextual_info: 股票收盘价以公开发行股票并上市之日为基准向后复权计算
    """
    df = df.copy()
    
    # 初始化标记列
    df['meu_12_5_subject'] = False
    df['meu_12_5_condition'] = False
    df['meu_12_5_constraint'] = None
    
    # ================== SUBJECT CHECK ==================
    # 1. 获取各公司上市日期和发行价格（保留原始上市日期）
    company_ipo_info = df.groupby('公司简称').agg(
        ipo_date=('上市日期', 'first'),  # 改名为ipo_date避免冲突
        issue_price=('发行价格', 'first')  
    ).reset_index()
    
    # 2. 找出各公司上市时的第一大股东
    major_shareholders = {}
    for _, row in company_ipo_info.iterrows():
        company = row['公司简称']
        ipo_date = row['ipo_date']
        
        # 获取上市当天的股东数据
        ipo_day_data = df[(df['公司简称'] == company) & (df['日期'] == ipo_date)]
        
        if not ipo_day_data.empty:
            max_ratio = ipo_day_data['持股比例'].max()
            if max_ratio >= 0.05:
                valid_shareholders = ipo_day_data[ipo_day_data['持股比例'] == max_ratio]['股东'].unique()
                major_shareholders[company] = list(valid_shareholders)
    
    # 3. 标记有效subject
    df['meu_12_5_subject'] = df.apply(
        lambda x: x['股东'] in major_shareholders.get(x['公司简称'], []),
        axis=1
    )
    
    # ================== CONDITION CHECK ==================
    # 1. 检查上市时无控股股东/实际控制人
    # 使用之前保存的ipo_date
    ipo_day_data = df[df.apply(
        lambda x: x['日期'] == company_ipo_info.set_index('公司简称').loc[x['公司简称'], 'ipo_date'],
        axis=1
    )]
    
    no_control_companies = ipo_day_data.groupby('公司简称').apply(
        lambda g: not g['股东身份'].isin(['控股股东', '实际控制人']).any()
    )
    df['company_no_control'] = df['公司简称'].map(no_control_companies)
    
    # 2. 验证减持方式
    valid_method = df['减持方式'].isin(['竞价交易', '大宗交易'])
    
    # 3. 首次减持计划标记
    df.sort_values(['公司简称', '股东', '日期'], inplace=True)
    df['plan_seq'] = df.groupby(['公司简称', '股东'])['存在减持计划'].cumsum()
    first_plan = df['存在减持计划'] & (df['plan_seq'] == 1)
    
    df['meu_12_5_condition'] = df['company_no_control'] & valid_method & first_plan
    
    # ================== CONSTRAINT CHECK ==================
    # 验证20日收盘价不低于发行价（包含不足20天的情况）

    # 1. 计算复权价格
    df['复权收盘价'] = df['原始收盘价'] * df['复权因子']

    # 2. 获取发行价格和上市日期
    df = df.merge(
        company_ipo_info[['公司简称', 'issue_price', 'ipo_date']],
        on='公司简称',
        how='left'
    )

    # 3. 标记上市后的交易日（不含上市当天）
    df['is_post_ipo'] = df['日期'] > df['ipo_date']

    # 4. 核心逻辑：检查过去N天价格（N<=20）
    def check_price_window(group):
        group = group.sort_values('日期')
        post_ipo = group[group['is_post_ipo']].copy()
        
        results = []
        for idx, row in post_ipo.iterrows():
            # 获取有效价格窗口（包含IPO当天）
            date_cutoff = row['日期'] - pd.Timedelta(days=20)
            past_days = group[
                (group['日期'] < row['日期']) & 
                (group['日期'] >= max(date_cutoff, group['ipo_date'].iloc[0]))
            ]
            
            # 核心改进点：处理NaN值
            valid_prices = past_days['复权收盘价'].dropna()
            
            if not valid_prices.empty:
                # 存在有效价格时检查最小值
                min_price = valid_prices.min()
                results.append(min_price >= row['issue_price'])
            else:
                # 无有效价格时视为合规（包括全NaN和空窗口）
                results.append(True)
        
        post_ipo['price_valid'] = results
        return post_ipo

    # 5. 应用处理
    price_valid_df = df.groupby('公司简称', group_keys=False).apply(check_price_window)
    df = df.merge(
        price_valid_df[['price_valid']],
        left_index=True,
        right_index=True,
        how='left'
    )

    # 6. 最终标记
    df['meu_12_5_constraint'] = np.where(
        ~df['is_post_ipo'],  # 上市前
        np.nan,             # 无意义
        df['price_valid']   # 上市后结果（已处理不足20天情况）
    )

    # # 7. 清理中间列
    # df.drop(['is_post_ipo', 'price_valid', '复权收盘价', 'issue_price', 'ipo_date'],
    #         axis=1, inplace=True, errors='ignore')
    
    return df

In [ ]:
df = check_meu_12_5(df)
df[df['meu_12_5_condition']==True][['股东身份', '持股比例', 'meu_12_5_subject', 'meu_12_5_constraint']]  # [df['meu_12_5_constraint']==False] # [:30]     # [['股东身份']]

In [ ]:
df[df['meu_12_5_constraint']==False][['股东身份', '持股比例', 'meu_12_5_subject', 'meu_12_5_constraint', '复权收盘价', '发行价格', '日期', '计划披露日']]

In [ ]:
import matplotlib.pyplot as plt

# Create a new plotting window with a larger figure size
plt.figure(figsize=(14, 8))  # 调整图像大小

# Plot the first data series: Adjusted Closing Price
plt.plot(df['日期'], df['复权收盘价'], label='Adjusted Close Price')

# Plot the second data series: Issuing Price
plt.plot(df['日期'], df['发行价格'], label='Issuing Price')

# Highlight the portions where meu_12_5_constraint == False
constraint_violations = df[df['meu_12_5_constraint'] == False]
plt.scatter(constraint_violations['日期'], constraint_violations['复权收盘价'], 
            color='red', label='Constraint Violated', s=10, zorder=3)  # 缩小点的大小 (s=10)

# Add title and axis labels
plt.title('Adjusted Close Price & Issuing Price', fontsize=16)  # 增大标题字体
plt.xlabel('Date', fontsize=12)  # x轴为 "日期"
plt.ylabel('Price', fontsize=12)

# Show legend
plt.legend(fontsize=10, loc='upper left')  # 调整图例字体大小

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Add gridlines for better visualization
plt.grid(visible=True, linestyle='--', alpha=0.5)

# Optimize layout to avoid overlapping
plt.tight_layout()

# Display the plot
plt.show()

---

# Law Article 13

## MEU_13_2


| 字段 | 内容 |
|------|------|
| subject | 大股东 |
| condition | 通过协议转让方式减持股份导致出让方不再具有大股东身份, 且处在减持后6个月内 |
| constrain | 继续遵守本指引第四条规定 |
| contextual_info | nan |
| note | nan |
| relation | should_include |
| target | Law_4 |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1709 |
| completion_tokens | 7409 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_13_2(df):
    '''
    检查MEU_13_2合规性：
    subject: 大股东(涉及到主体变更的, subject默认为True, 依赖condition定位)
    condition: 通过协议转让减持导致失去大股东身份且在减持后6个月内
    constraint: 继续遵守第四条(本函数不做处理)
    '''
    df = df.copy()
    
    # 初始化标记列
    df['meu_13_2_subject'] = True # 涉及到身份转换的条目, subject默认正确, 因为已经蕴含在condition之中
    df['meu_13_2_condition'] = False
    df['meu_13_2_constraint'] = None

    # 1. 标记subject有效性
    pass

    # 2. 标记condition有效性（分组处理股东减持事件）
    for (company, shareholder), group in df.groupby(['公司简称', '股东']):
        # 按日期排序，确保前后记录顺序正确
        group = group.sort_values('日期')
        
        # 计算前一日持股比例（使用 shift(1) 获取前一天的值）
        group['前一日持股比例'] = group['持股比例'].shift(1)
        group['前一日股东身份'] = group['股东身份'].shift(1)


        last_day_major = (
            (group['前一日股东身份'].isin(['控股股东', '实际控制人', '持股5%以上股东']))
            | (group['前一日持股比例'] >= 0.05)
        )

        now_day_major = (
            (group['股东身份'].isin(['控股股东', '实际控制人', '持股5%以上股东']))
            | (group['持股比例'] >= 0.05)
        )

        # 筛选触发条件：
        # 1. 当日减持方式为协议转让
        # 2. 前一日持股比例 >=5%
        # 3. 当日持股比例 <5%
        trigger_dates = group[
            (group['减持方式'] == '协议转让') &
            (last_day_major) &
            (~now_day_major)
        ]['日期']

        print(f"协议转让导致失去大股东身份触发记录数: {len(trigger_dates)}")
        
        # 对每个触发日期标记后续180天（不含触发当天）
        for trigger_date in trigger_dates:
            start_date = trigger_date + pd.DateOffset(days=1)
            end_date = trigger_date + pd.DateOffset(days=180)
            
            mask = (
                (df['公司简称'] == company) &
                (df['股东'] == shareholder) &
                (df['日期'].between(start_date, end_date, inclusive='both'))
            )
            
            df.loc[mask, 'meu_13_2_condition'] = True

    # 3. 标记constraint有效性
    df['meu_13_2_constraint'] = None 

    return df

In [ ]:
df = check_meu_13_2(df)
df[df['meu_13_2_condition'] == True]

In [ ]:
from datetime import datetime

# 定义两个日期
date1 = datetime.strptime("2023-03-22", "%Y-%m-%d")
date2 = datetime.strptime("2022-09-26", "%Y-%m-%d")

# 计算两个日期之间的自然日差
days_difference = (date1 - date2).days

print(f"两个日期之间的自然日差是 {days_difference} 天.")

In [ ]:
df[df['meu_13_2_condition'] == True][['日期', '股东', '公司简称', '股东身份', '持股比例', '减持方式']]

In [ ]:
df[['日期', '股东', '公司简称', '股东身份', '持股比例', '减持方式']]['持股比例'].plot()

---

## MEU_13_3


| 字段 | 内容 |
|------|------|
| subject | 控股股东 | 实际控制人 |
| condition | 通过协议转让方式减持股份导致其不再具有控股股东、实际控制人身份, 且处于减持后的6个月内 |
| constrain | 应当继续遵守本指引第十二条第一款第二、三项规定 |
| contextual_info | nan |
| note | nan |
| relation | should_include |
| target | MEU_12_1;MEU_12_2 |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1723 |
| completion_tokens | 9385 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_13_3(df):
    '''
    检查MEU_13_3的合规性：
    subject: 控股股东 | 实际控制人
    condition: 通过协议转让减持导致失去身份，且在减持后的6个月内
    constraint: 继续遵守第十二条相关规定（假设检查减持后的六个月内是否有减持行为）
    '''
    df = df.copy()
    
    # 初始化标记列
    df['meu_13_3_subject'] = True # 涉及到身份转换的条目, subject默认正确, 因为已经蕴含在condition之中
    df['meu_13_3_condition'] = False
    df['meu_13_3_constraint'] = None

    # 1. 标记subject有效性
    pass

    # 2. 标记condition有效性（分组处理股东减持事件）
    for (company, shareholder), group in df.groupby(['公司简称', '股东']):
        # 按日期排序，确保前后记录顺序正确
        group = group.sort_values('日期')
        
        # 计算前一日持股比例（使用 shift(1) 获取前一天的值）
        group['前一日持股比例'] = group['持股比例'].shift(1)
        group['前一日股东身份'] = group['股东身份'].shift(1)

        last_day_control = (
            (group['前一日股东身份'].isin(['控股股东', '实际控制人']))
        )

        now_day_control = (
            (group['股东身份'].isin(['控股股东', '实际控制人']))
        )

        trigger_dates = group[
            (group['减持方式'] == '协议转让') &
            (last_day_control) &
            (~now_day_control)
        ]['日期']

        print(f"协议转让导致失去控股股东, 实际控制人身份触发记录数: {len(trigger_dates)}")
        
        # 对每个触发日期标记后续180天（不含触发当天）
        for trigger_date in trigger_dates:
            start_date = trigger_date + pd.DateOffset(days=1)
            end_date = trigger_date + pd.DateOffset(days=180)
            
            mask = (
                (df['公司简称'] == company) &
                (df['股东'] == shareholder) &
                (df['日期'].between(start_date, end_date, inclusive='both'))
            )
            
            df.loc[mask, 'meu_13_3_condition'] = True

    # 3. 标记constraint有效性
    df['meu_13_3_constraint'] = None 

    return df
    

In [ ]:
df = check_meu_13_3(df)
df

---

# Law Article 14

## MEU_14_1


| 字段 | 内容 |
|------|------|
| subject | 上市公司董监高 |
| condition | 上市公司因涉嫌证券期货违法犯罪，在被中国证监会及其派出机构立案调查或者被司法机关立案侦查期间 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1710 |
| completion_tokens | 4302 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_14_1(df):
    '''
    检查MEU_14_1合规性：
    subject: 上市公司董监高
    condition: 公司因涉嫌证券期货违法犯罪被立案调查或侦查期间
    constraint: 不得减持股份
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_14_1_subject'] = False
    df['meu_14_1_condition'] = False
    df['meu_14_1_constraint'] = None

    # 1. 验证责任主体：股东身份为董监高
    valid_subject = df['股东身份'] == '董监高'
    df.loc[valid_subject, 'meu_14_1_subject'] = True

    # 2. 标记触发条件有效性（处于被调查/侦查期间）
    investigation_events = [
        '被中国证监会及其派出机构立案调查',
        '中国证监会及其派出机构立案调查结束',
        '被司法机关立案侦查',
        '司法机关立案侦查结束'
    ]

    # 获取所有涉及调查/侦查的（公司简称 + 股东）组合
    involved_cases = df[df['公司涉嫌证券期货违法犯罪事件'].isin(investigation_events)][
        ['公司简称', '股东']
    ].drop_duplicates()

    # 获取每个(公司,股东)的最后记录日期
    last_dates = df.groupby(['公司简称', '股东'])['日期'].max().reset_index()
    
    for _, case in involved_cases.iterrows():
        company = case['公司简称']
        shareholder = case['股东']
        
        # 获取该（公司+股东）的所有事件并按时间排序
        case_events = df[
            (df['公司简称'] == company) & 
            (df['股东'] == shareholder)
        ][['日期', '公司涉嫌证券期货违法犯罪事件']].dropna().sort_values('日期')

        # 跟踪当前是否处于调查/侦查期间
        in_investigation = False
        investigation_start = None

        for _, row in case_events.iterrows():
            event = row['公司涉嫌证券期货违法犯罪事件']
            date = row['日期']

            if event == '被中国证监会及其派出机构立案调查':
                in_investigation = True
                investigation_start = date
            elif event == '中国证监会及其派出机构立案调查结束' and in_investigation:
                # 标记从立案到结案期间的所有记录
                df.loc[
                    (df['公司简称'] == company) & 
                    (df['股东'] == shareholder) & 
                    (df['日期'] >= investigation_start) & 
                    (df['日期'] <= date),
                    'meu_14_1_condition'
                ] = True
                in_investigation = False
            elif event == '被司法机关立案侦查':
                in_investigation = True
                investigation_start = date
            elif event == '司法机关立案侦查结束' and in_investigation:
                # 标记从立案到结案期间的所有记录
                df.loc[
                    (df['公司简称'] == company) & 
                    (df['股东'] == shareholder) & 
                    (df['日期'] >= investigation_start) & 
                    (df['日期'] <= date),
                    'meu_14_1_condition'
                ] = True
                in_investigation = False
        
        # 处理调查开始但未结束的情况
        if in_investigation and investigation_start:
            last_date = last_dates[
                (last_dates['公司简称'] == company) & 
                (last_dates['股东'] == shareholder)
            ]['日期'].values[0]
            
            # 标记从立案到最后记录日期的所有记录
            df.loc[
                (df['公司简称'] == company) & 
                (df['股东'] == shareholder) & 
                (df['日期'] >= investigation_start) & 
                (df['日期'] <= last_date),
                'meu_14_1_condition'
            ] = True

    # 3. 验证约束条件：当日无减持行为
    # 处理空值并转换为数值型
    df['当日减持比例'] = pd.to_numeric(df['当日减持比例'], errors='coerce').fillna(0)
    valid_constraint = df['当日减持比例'] == 0

    # 设置约束标记
    df.loc[valid_constraint, 'meu_14_1_constraint'] = True
    df.loc[~valid_constraint, 'meu_14_1_constraint'] = False

    return df


In [ ]:
df = check_meu_14_1(df)
df

In [ ]:
df[268:305][['公司涉嫌证券期货违法犯罪事件', 'meu_14_1_condition']]

In [ ]:
df[df['meu_14_1_condition']==True][['公司涉嫌证券期货违法犯罪事件', 'meu_14_1_condition']]

---

## MEU_14_2


| 字段 | 内容 |
|------|------|
| subject | 上市公司董监高 |
| condition | 上市公司因涉嫌证券期货违法犯罪被中国证监会及其派出机构立案调查或者被司法机关立案侦查，在行政处罚决定、刑事判决作出之后未满6个月 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1721 |
| completion_tokens | 5117 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_14_2(df):
    '''
    合规性检查函数：MEU_14_2
    subject: 上市公司董监高
    condition: 公司因证券期货违法犯罪被立案调查/侦查且处罚决定/刑事判决未满6个月
    constraint: 不得减持股份
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_14_2_subject'] = False
    df['meu_14_2_condition'] = False
    df['meu_14_2_constraint'] = None

    # 1. 标记责任主体 (上市公司董监高)
    subject_mask = df['股东身份'] == '董监高'
    df.loc[subject_mask, 'meu_14_2_subject'] = True

    # 2. 标记触发条件 (处罚后未满6个月)
    # 筛选处罚相关事件
    penalty_events = df[df['公司涉嫌证券期货违法犯罪事件'].isin(['行政处罚决定作出', '刑事判决作出'])]
    
    # 按公司获取最新处罚日期
    latest_penalty = penalty_events.groupby('公司简称')['公告日期'].max().reset_index()
    latest_penalty.rename(columns={'公告日期':'latest_penalty_date'}, inplace=True)
    
    # 合并处罚日期到主表
    df = df.merge(latest_penalty, on='公司简称', how='left')
    
    # 计算自然日间隔
    if 'latest_penalty_date' in df.columns:
        df['penalty_days'] = (df['日期'] - df['latest_penalty_date']).dt.days
        condition_mask = (df['penalty_days'] >= 0) & (df['penalty_days'] <= 180)
        df.loc[condition_mask, 'meu_14_2_condition'] = True

    # 3. 标记约束条件 (不得减持)
    df['meu_14_2_constraint'] = df['当日减持比例'] <= 0  # 无减持为合规
    
    # 清理辅助列
    df.drop(['latest_penalty_date', 'penalty_days'], axis=1, errors='ignore', inplace=True)
    
    return df

In [ ]:
df = check_meu_14_2(df)
df

---

## MEU_14_3


| 字段 | 内容 |
|------|------|
| subject | 上市公司董监高 |
| condition | 本人因涉嫌与该上市公司有关的证券期货违法犯罪，在被中国证监会及其派出机构立案调查或者被司法机关立案侦查期间 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1713 |
| completion_tokens | 3719 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_14_3(df):
    '''
    检查MEU_14_3合规性：
    subject: 上市公司董监高
    condition: 股东因涉嫌证券期货违法犯罪被立案调查或侦查期间
    constraint: 不得减持股份
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_14_3_subject'] = False
    df['meu_14_3_condition'] = False
    df['meu_14_3_constraint'] = None

    # 1. 验证责任主体：股东身份为董监高
    valid_subject = df['股东身份'] == '董监高'
    df.loc[valid_subject, 'meu_14_3_subject'] = True

    # 2. 标记触发条件有效性（处于被调查/侦查期间）
    investigation_events = [
        '被中国证监会及其派出机构立案调查',
        '中国证监会及其派出机构立案调查结束',
        '被司法机关立案侦查',
        '司法机关立案侦查结束'
    ]

    # 获取所有涉及调查/侦查的（公司简称 + 股东）组合
    involved_cases = df[df['股东涉嫌证券期货违法犯罪事件'].isin(investigation_events)][
        ['公司简称', '股东']
    ].drop_duplicates()

    # 获取每个(公司,股东)的最后记录日期
    last_dates = df.groupby(['公司简称', '股东'])['日期'].max().reset_index()
    
    for _, case in involved_cases.iterrows():
        company = case['公司简称']
        shareholder = case['股东']
        
        # 获取该（公司+股东）的所有事件并按时间排序
        case_events = df[
            (df['公司简称'] == company) & 
            (df['股东'] == shareholder)
        ][['日期', '股东涉嫌证券期货违法犯罪事件']].dropna().sort_values('日期')

        # 跟踪当前是否处于调查/侦查期间
        in_investigation = False
        investigation_start = None

        for _, row in case_events.iterrows():
            event = row['股东涉嫌证券期货违法犯罪事件']
            date = row['日期']

            if event == '被中国证监会及其派出机构立案调查':
                in_investigation = True
                investigation_start = date
            elif event == '中国证监会及其派出机构立案调查结束' and in_investigation:
                # 标记从立案到结案期间的所有记录
                df.loc[
                    (df['公司简称'] == company) & 
                    (df['股东'] == shareholder) & 
                    (df['日期'] >= investigation_start) & 
                    (df['日期'] <= date),
                    'meu_14_3_condition'
                ] = True
                in_investigation = False
            elif event == '被司法机关立案侦查':
                in_investigation = True
                investigation_start = date
            elif event == '司法机关立案侦查结束' and in_investigation:
                # 标记从立案到结案期间的所有记录
                df.loc[
                    (df['公司简称'] == company) & 
                    (df['股东'] == shareholder) & 
                    (df['日期'] >= investigation_start) & 
                    (df['日期'] <= date),
                    'meu_14_3_condition'
                ] = True
                in_investigation = False
        
        # 处理调查开始但未结束的情况
        if in_investigation and investigation_start:
            last_date = last_dates[
                (last_dates['公司简称'] == company) & 
                (last_dates['股东'] == shareholder)
            ]['日期'].values[0]
            
            # 标记从立案到最后记录日期的所有记录
            df.loc[
                (df['公司简称'] == company) & 
                (df['股东'] == shareholder) & 
                (df['日期'] >= investigation_start) & 
                (df['日期'] <= last_date),
                'meu_14_3_condition'
            ] = True

    # 3. 验证约束条件：当日无减持行为
    # 处理空值并转换为数值型
    df['当日减持比例'] = pd.to_numeric(df['当日减持比例'], errors='coerce').fillna(0)
    valid_constraint = df['当日减持比例'] == 0

    # 设置约束标记
    df.loc[valid_constraint, 'meu_14_3_constraint'] = True
    df.loc[~valid_constraint, 'meu_14_3_constraint'] = False

    return df

In [ ]:
df = check_meu_14_3(df)
df

---

## MEU_14_4


| 字段 | 内容 |
|------|------|
| subject | 上市公司董监高 |
| condition | 本人因涉嫌与该上市公司有关的证券期货违法犯罪被中国证监会及其派出机构立案调查或者被司法机关立案侦查，在行政处罚决定、刑事判决作出之后未满6个月 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1724 |
| completion_tokens | 6450 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_14_4(df):
    '''
    合规性检查函数: MEU_12_4
    subject: 董监高
    condition: 股东因证券期货违法犯罪被立案调查/侦查且处罚决定/刑事判决未满6个月
    constraint: 不得减持股份
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_14_4_subject'] = False
    df['meu_14_4_condition'] = False
    df['meu_14_4_constraint'] = None

    # 1. 标记责任主体
    df['meu_14_4_subject'] = df['股东身份'] == '董监高'

    # 2. 标记触发条件 (处罚后未满6个月)
    # 筛选股东处罚相关事件
    penalty_events = df[df['股东涉嫌证券期货违法犯罪事件'].isin(['行政处罚决定作出', '刑事判决作出'])]
    
    # 按股东获取最新处罚日期
    latest_penalty = penalty_events.groupby('股东')['公告日期'].max().reset_index()
    latest_penalty.rename(columns={'公告日期':'latest_penalty_date'}, inplace=True)
    
    # 合并处罚日期到主表
    df = df.merge(latest_penalty, on='股东', how='left')
    
    # 计算自然日间隔
    if 'latest_penalty_date' in df.columns:
        df['penalty_days'] = (df['日期'] - df['latest_penalty_date']).dt.days
        condition_mask = (df['penalty_days'] >= 0) & (df['penalty_days'] <= 180)
        df.loc[condition_mask, 'meu_12_4_condition'] = True

    # 3. 标记约束条件 (不得减持)
    df['meu_14_4_constraint'] = df['当日减持比例'] <= 0  # 无减持为合规
    
    # 清理辅助列
    df.drop(['latest_penalty_date', 'penalty_days'], axis=1, errors='ignore', inplace=True)
    
    return df

In [ ]:
df = check_meu_14_4(df)
df

---

## MEU_14_5


| 字段 | 内容 |
|------|------|
| subject | 上市公司董监高 |
| condition | 本人因涉及证券期货违法，被中国证监会行政处罚，罚没款尚未足额缴纳，且不存在法律、行政法规另有规定或减持资金用于缴纳罚没款的情况 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | 不考虑法律、行政法规另有规定或减持资金用于缴纳罚没款的情况 |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1740 |
| completion_tokens | 4183 |


### 代码实现

In [ ]:
# # 读取模拟数据
# import pandas as pd
# df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
# date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
# for col in date_columns:
#     df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
# import pandas as pd

# def check_meu_14_5(df):
#     '''
#     检查MEU_14_5合规性：
#     "subject": "上市公司董监高",
#     "condition": "被证监会行政处罚且罚没款未缴",
#     "constraint": "不得减持股份",
#     实现逻辑：
#     1. subject检查：股东身份为董监高
#     2. condition检查：存在行政处罚决定且未缴款（通过事件字段判断）
#     3. constraint检查：当日无减持行为（通过减持比例判断）
#     '''
#     df = df.copy()

#     # 初始化合规标记列
#     df['meu_14_5_subject'] = False
#     df['meu_14_5_condition'] = False
#     df['meu_14_5_constraint'] = None

#     # 1. 验证责任主体（上市公司董监高）
#     subject_mask = df['股东身份'] == '董监高'
#     df.loc[subject_mask, 'meu_14_5_subject'] = True

#     # 2. 验证触发条件（存在未缴款的行政处罚）
#     # 根据数据字段直接匹配行政处罚决定状态
#     condition_mask = df['股东涉嫌证券期货违法犯罪事件'] == '行政处罚决定作出'
#     df.loc[condition_mask, 'meu_14_5_condition'] = True

#     # 3. 验证约束条件（未发生减持行为）
#     # 通过当日减持比例判断，0表示无减持
#     constraint_mask = df['当日减持比例'] == 0
#     df.loc[constraint_mask, 'meu_14_5_constraint'] = True
#     df.loc[~constraint_mask, 'meu_14_5_constraint'] = False

#     return df

In [ ]:
# df = check_meu_14_5(df)
# df

---

## MEU_14_6


| 字段 | 内容 |
|------|------|
| subject | 上市公司董监高 |
| condition | 本人因涉及与本上市公司有关的违法违规，被证券交易所公开谴责未满三个月 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1704 |
| completion_tokens | 5533 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_14_6(df):
    '''
    检查MEU_14_6合规性（基于11_3风格重写）：
    "subject": "上市公司董监高", 
    "condition": "被交易所公开谴责未满三个月(90个自然日)", 
    "constraint": "不得减持股份"
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_14_6_subject'] = False
    df['meu_14_6_condition'] = False
    df['meu_14_6_constraint'] = None

    # 1. 标记责任主体（董监高）
    df['meu_14_6_subject'] = df['股东身份'] == '董监高'

    # 2. 处理公开谴责时间窗口（按股东追踪）
    condemn_events = df[df['股东涉嫌证券期货违法犯罪事件'] == '被本所公开谴责']
    
    if not condemn_events.empty:
        # 按股东获取谴责日期和窗口结束日期
        condemn_windows = condemn_events.groupby(['公司简称', '股东'])['日期'].agg([
            ('condemn_date', 'min'),  # 最早谴责日期
            ('window_end', lambda x: x.min() + pd.Timedelta(days=90))  # 90天后
        ]).reset_index()
        
        # 为每个股东创建时间窗口标记
        for _, row in condemn_windows.iterrows():
            window_mask = (
                (df['公司简称'] == row['公司简称']) &
                (df['股东'] == row['股东']) &
                (df['日期'] >= row['condemn_date']) &
                (df['日期'] <= row['window_end'])
            )
            df.loc[window_mask, 'meu_14_6_condition'] = True

    # 3. 处理减持约束（NaN视为合规）
    df['meu_14_6_constraint'] = df['当日减持比例'] <= 0

    return df

In [ ]:
df = check_meu_14_6(df)
df

In [ ]:
df[df['meu_14_6_condition']==True][['日期']]

In [ ]:
df[['日期', '公司简称', '股东涉嫌证券期货违法犯罪事件','meu_14_6_condition']][75:138]

---

## MEU_14_7


| 字段 | 内容 |
|------|------|
| subject | 上市公司董监高 |
| condition | 上市公司股票因可能触及重大违法强制退市情形而被本所实施退市风险警示，且处于本所规定的限制转让的期限内 |
| constrain | 不得减持其所持有的本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1718 |
| completion_tokens | 8122 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_14_7(df):
    '''
    检查MEU_14_7合规性：
    subject: 上市公司董监高
    condition: 公司股票因重大违法强制退市风险被实施退市风险警示且处于限制转让期
    constraint: 不得减持所持股份
    
    实现逻辑：
    1. subject标记：股东身份为"董监高"
    2. condition标记：公司存在重大违法调查且处于限制转让期
    3. constraint标记：当日无减持行为
    
    改进点：处理"退市风险警示"列中的'限制转让期开始'和'限制转让期结束'事件，
            标记这两个日期之间的记录为condition有效
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_14_7_subject'] = False
    df['meu_14_7_condition'] = False
    df['meu_14_7_constraint'] = None

    # 1. 标记责任主体有效性（董监高）
    valid_subject = df['股东身份'] == '董监高'
    df.loc[valid_subject, 'meu_14_7_subject'] = True

    # 2. 标记触发条件有效性（处于限制转让期内）
    restriction_events = [
        '限制转让期开始',
        '限制转让期结束'
    ]

    # 获取所有涉及限制转让的（公司简称 + 股东）组合
    involved_cases = df[df['退市风险警示'].isin(restriction_events)][
        ['公司简称', '股东']
    ].drop_duplicates()

    # 获取每个(公司,股东)的最后记录日期
    last_dates = df.groupby(['公司简称', '股东'])['日期'].max().reset_index()
    
    for _, case in involved_cases.iterrows():
        company = case['公司简称']
        shareholder = case['股东']
        
        # 获取该（公司+股东）的所有退市风险警示事件并按时间排序
        case_events = df[
            (df['公司简称'] == company) & 
            (df['股东'] == shareholder) & 
            (df['退市风险警示'].isin(restriction_events))
        ][['日期', '退市风险警示']].dropna().sort_values('日期')

        # 跟踪当前是否处于限制转让期间
        in_restriction = False
        restriction_start = None

        for _, row in case_events.iterrows():
            event = row['退市风险警示']
            date = row['日期']

            if event == '限制转让期开始':
                in_restriction = True
                restriction_start = date
            elif event == '限制转让期结束' and in_restriction:
                # 标记从开始到结束期间的所有记录
                df.loc[
                    (df['公司简称'] == company) & 
                    (df['股东'] == shareholder) & 
                    (df['日期'] >= restriction_start) & 
                    (df['日期'] <= date),
                    'meu_14_7_condition'
                ] = True
                in_restriction = False
        
        # 处理限制期开始但未结束的情况
        if in_restriction and restriction_start:
            last_date = last_dates[
                (last_dates['公司简称'] == company) & 
                (last_dates['股东'] == shareholder)
            ]['日期'].values[0]
            
            # 标记从开始到最后记录日期的所有记录
            df.loc[
                (df['公司简称'] == company) & 
                (df['股东'] == shareholder) & 
                (df['日期'] >= restriction_start) & 
                (df['日期'] <= last_date),
                'meu_14_7_condition'
            ] = True

    # 3. 标记约束有效性（未减持股份）
    df['meu_14_7_constraint'] = (df['当日减持比例'] == 0)

    return df

In [ ]:
df = check_meu_14_7(df)
df

In [ ]:
len(df) - df['meu_14_7_constraint'].sum()

---

# Law Article 16

## MEU_16_1


| 字段 | 内容 |
|------|------|
| subject | 上市公司董监高 |
| condition | 上市公司年度报告、半年度报告公告前15日内 |
| constrain | 不得买卖本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1698 |
| completion_tokens | 4784 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_16_1(df):
    '''
    检查MEU_16_1合规性：
    - subject: 上市公司董监高
    - condition: 处于年报/半年报公告前15日内
    - constraint: 当日无股份买卖行为
    '''
    df = df.copy()
    
    # 初始化标记列
    df['meu_16_1_subject'] = False
    df['meu_16_1_condition'] = False
    df['meu_16_1_constraint'] = None

    # 1. 验证责任主体（独立检查）
    valid_subject = df['股东身份'] == '董监高'
    df.loc[valid_subject, 'meu_16_1_subject'] = True

    # 2. 验证触发条件（按公司和股东分组检查）
    # 构建(公司,股东)报告日期字典（年报/半年报）
    report_dates = df[df['公告类型'].isin(['年报', '半年报'])]
    company_holder_report_dates = report_dates.groupby(['公司简称', '股东'])['公告日期'].apply(lambda x: x.unique().tolist()).to_dict()
    
    # 定义自然日窗口检查函数
    def check_report_window(row):
        key = (row['公司简称'], row['股东'])
        current_date = row['日期']
        for report_date in company_holder_report_dates.get(key, []):
            if (report_date - pd.Timedelta(days=15)) <= current_date < report_date:
                return True
        return False
    
    valid_condition = df.apply(check_report_window, axis=1)
    df.loc[valid_condition, 'meu_16_1_condition'] = True

    # 3. 验证约束内容（独立检查）
    # 通过当日减持比例判断是否发生交易
    valid_constraint = df['当日减持比例'] == 0
    df.loc[valid_constraint, 'meu_16_1_constraint'] = True
    df.loc[~valid_constraint, 'meu_16_1_constraint'] = False

    return df

In [ ]:
df = check_meu_16_1(df)
df

In [ ]:
df[df['meu_16_1_condition']==True]

In [ ]:
df[['日期', '股东', '公告类型', '公告日期', 'meu_16_1_condition']][63:90]

---

## MEU_16_2


| 字段 | 内容 |
|------|------|
| subject | 上市公司董监高 |
| condition | 上市公司季度报告、业绩预告、业绩快报公告前5日内 |
| constrain | 不得买卖本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1701 |
| completion_tokens | 6500 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_16_2(df):
    '''
    检查MEU_16_2合规性：
    - subject: 上市公司董监高
    - condition: 处于季报公告前5日内
    - constraint: 当日无股份买卖行为
    '''
    df = df.copy()
    
    # 初始化标记列
    df['meu_16_2_subject'] = False
    df['meu_16_2_condition'] = False
    df['meu_16_2_constraint'] = None

    # 1. 验证责任主体（独立检查）
    valid_subject = df['股东身份'] == '董监高'
    df.loc[valid_subject, 'meu_16_2_subject'] = True

    # 2. 验证触发条件（按公司和股东分组检查）
    # 构建(公司,股东)报告日期字典（季报）
    report_dates = df[df['公告类型'].isin(['季报'])]
    company_holder_report_dates = report_dates.groupby(['公司简称', '股东'])['公告日期'].apply(lambda x: x.unique().tolist()).to_dict()
    
    # 定义自然日窗口检查函数（5日窗口）
    def check_report_window(row):
        key = (row['公司简称'], row['股东'])
        current_date = row['日期']
        for report_date in company_holder_report_dates.get(key, []):
            if (report_date - pd.Timedelta(days=5)) <= current_date < report_date:
                return True
        return False
    
    valid_condition = df.apply(check_report_window, axis=1)
    df.loc[valid_condition, 'meu_16_2_condition'] = True

    # 3. 验证约束内容（独立检查）
    # 通过当日减持比例判断是否发生交易
    valid_constraint = df['当日减持比例'] == 0
    df.loc[valid_constraint, 'meu_16_2_constraint'] = True
    df.loc[~valid_constraint, 'meu_16_2_constraint'] = False

    return df

In [ ]:
df = check_meu_16_2(df)
df

In [ ]:
df[df['meu_16_2_condition']==True]

In [ ]:
df[['日期', '股东', '公告类型', '公告日期', 'meu_16_2_condition']][113:120]

---

# Law Article 17

## MEU_17_1


| 字段 | 内容 |
|------|------|
| subject | 上市公司董监高, 以及离任六个月内的董监高 |
| condition | 在其就任时确定的任期内和任期届满后6个月内，每年通过集中竞价、大宗交易、协议转让等方式转让股份且不属于因司法强制执行、继承、遗赠、依法分割财产等导致股份变动 |
| constrain | 每年转让的股份不得超过其所持本公司股份总数的25% |
| contextual_info | nan |
| note | nan |
| relation | refer_to |
| target | MEU_18_1;MEU_18_2;MEU_18_3;MEU_18_4 |
| type | 实际执行单元 |
| comments | 不考虑因司法强制执行、继承、遗赠、依法分割财产等导致股份变动的情况 |
| prompt_tokens | 1821 |
| completion_tokens | 7230 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_17_1(df):
    '''
    验证MEU_17_1合规性：
    subject: 上市公司董监高 | 离任六个月内的董监高
    condition: 通过集中竞价、大宗交易、协议转让等方式转让股份且不属于因司法强制执行、继承、遗赠、依法分割财产等导致股份变动
    constraint: 每年转让的股份不得超过其所持本公司股份总数的25%
    contextual_info: 以上年末其所持股总数作为基数计算可转让股份数量
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_17_1_subject'] = False
    df['meu_17_1_condition'] = False
    df['meu_17_1_constraint'] = None

    # ===================== 预处理 =====================
    # 转换日期格式并提取年份
    df['日期'] = pd.to_datetime(df['日期'])
    df['离任日期'] = pd.to_datetime(df['离任日期'])
    df['year'] = df['日期'].dt.year  # 新增全局年份列

    # ===================== 标记subject =====================
    # 时间条件（任期内或离任后6个月, 按180天计算）
    time_cond = (
        df['离任日期'].isna() |  # 在任情况
        (df['日期'] <= df['离任日期'] + pd.Timedelta(days=180))  # 离任后180天
    )
    # 主体条件
    valid_subject = (df['股东身份'] == '董监高') & time_cond
    df.loc[valid_subject, 'meu_17_1_subject'] = True

    # ===================== 标记condition =====================
    # 处理所有数据（不再筛选subject）
    df_sorted = df.sort_values(['公司简称', '股东', '日期']).copy()
    
    # 计算减持比例
    df_sorted['prev_持股比例'] = df_sorted.groupby(['公司简称', '股东'])['持股比例'].shift(1)
    df_sorted['当日减持比例'] = (df_sorted['prev_持股比例'] - df_sorted['持股比例']).clip(lower=0)
    
    # 条件检查（保留注释的代码）
    method_cond = df_sorted['减持方式'].isin(['竞价交易', '大宗交易', '协议转让'])
    # reason_cond = ~df_sorted['拟减持原因'].isin(['司法强制执行', '继承', '遗赠', '依法分割财产'])
    has_daily_reduction = (df_sorted['当日减持比例'] > 0)
    
    # 创建中间条件列
    df_sorted['condition_met'] = has_daily_reduction & method_cond # & reason_cond
    
    # 按分组标记condition（所有数据参与计算）
    df_sorted['group_condition'] = df_sorted.groupby(
        ['公司简称', '股东', 'year']
    )['condition_met'].transform('any')

    # 回写结果到原DF
    df.loc[df_sorted.index, 'meu_17_1_condition'] = df_sorted['group_condition']

    # ===================== 标记constraint =====================
    # 添加年份列和初始化中间变量列
    df['year'] = df['日期'].dt.year
    df['prev_share'] = None  # 上年末持股比例列
    df['total_reduction'] = None  # 当年累计减持比例列

    # 按分组处理数据
    for (company, shareholder), group in df.groupby(['公司简称', '股东']):
        group_sorted = group.sort_values('日期')
        years = group_sorted['year'].unique()
        
        for year in years:
            current_year_data = group_sorted[(group_sorted['year'] == year)]  # 先不筛选日期，用于获取IPO日期
            ipo_date = current_year_data['上市日期'].iloc[0]
            
            # 筛选当前年份且日期在上市后的数据
            current_year_data = group_sorted[
                (group_sorted['year'] == year) & 
                (group_sorted['year'] >= ipo_date.year)
            ]
            if current_year_data.empty:
                continue
            
            max_date = current_year_data['日期'].max()
            previous_year = year - 1
            prev_year_data = group_sorted[group_sorted['year'] == previous_year]
            
            if not prev_year_data.empty:
                last_prev_date = prev_year_data['日期'].max()
                prev_share_row = prev_year_data[prev_year_data['日期'] == last_prev_date]
                
                if not prev_share_row.empty:
                    prev_share = prev_share_row['持股比例'].values[0]
                    total_reduction = current_year_data['当日减持比例'].sum()
                    
                    # 写入中间变量
                    df.loc[current_year_data.index, 'prev_share'] = prev_share
                    df.loc[current_year_data.index, 'total_reduction'] = total_reduction
                    
                    # 设置约束条件
                    if total_reduction <= prev_share * 0.25:
                        # 符合规则
                        df.loc[current_year_data.index, 'meu_17_1_constraint'] = True
                    else: 
                        # 不符合规则
                        df.loc[current_year_data.index, 'meu_17_1_constraint'] = False

    return df


In [ ]:
df = check_meu_17_1(df)
df.to_csv("temp_meu_17_1.csv", encoding='utf-8-sig', index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 读取数据
df = pd.read_csv("data_simulation/data_generated/data_simulate_08.csv")

# 转换日期格式
df['日期'] = pd.to_datetime(df['日期'])
df['上市日期'] = pd.to_datetime(df['上市日期'])

# 添加年份列和初始化中间变量列
df['year'] = df['日期'].dt.year
df['constraint'] = None
df['prev_share'] = None  # 新增上年末持股比例列
df['total_reduction'] = None  # 新增当年累计减持比例列

# 按分组处理数据
for (company, shareholder), group in df.groupby(['公司简称', '股东']):
    group_sorted = group.sort_values('日期')
    years = group_sorted['year'].unique()
    
    for year in years:
        current_year_data = group_sorted[(group_sorted['year'] == year)]  # 先不筛选日期，用于获取IPO日期
        ipo_date = current_year_data['上市日期'].iloc[0]
        
        # 筛选当前年份且日期在上市后的数据
        current_year_data = group_sorted[
            (group_sorted['year'] == year) & 
            (group_sorted['year'] >= ipo_date.year)
        ]
        if current_year_data.empty:
            continue
        
        max_date = current_year_data['日期'].max()
        previous_year = year - 1
        prev_year_data = group_sorted[group_sorted['year'] == previous_year]
        
        if not prev_year_data.empty:
            last_prev_date = prev_year_data['日期'].max()
            prev_share_row = prev_year_data[prev_year_data['日期'] == last_prev_date]
            
            if not prev_share_row.empty:
                prev_share = prev_share_row['持股比例'].values[0]
                total_reduction = current_year_data['当日减持比例'].sum()
                
                # 写入中间变量
                df.loc[current_year_data.index, 'prev_share'] = prev_share
                df.loc[current_year_data.index, 'total_reduction'] = total_reduction
                
                # 设置约束条件
                if total_reduction <= prev_share * 0.25:
                    # 符合规则
                    df.loc[current_year_data.index, 'constraint'] = True
                else: 
                    # 不符合规则
                    df.loc[current_year_data.index, 'constraint'] = False

# 可视化部分
for (company, shareholder), group in df.groupby(['公司简称', '股东']):
    group = group.sort_values('日期').dropna(subset=['持股比例'])
    if group.empty:
        continue
        
    plt.figure(figsize=(12, 6))
    ax = plt.gca()
    
    # 绘制持股比例折线
    ax.plot(group['日期'], group['持股比例'], 
            marker='o', linestyle='-', label='每日持股比例')
    
    # 标记关键数据和约束状态
    years = group['year'].unique()
    for year in years:
        year_data = group[group['year'] == year]
        if year_data.empty:
            continue
            
        # 获取关键数据
        prev_share = year_data['prev_share'].iloc[0]
        total_reduction = year_data['total_reduction'].iloc[0]
        constraint = year_data['constraint'].iloc[0] if not pd.isna(year_data['constraint'].iloc[0]) else None
        date_range = (year_data['日期'].min(), year_data['日期'].max())
        
        # 绘制上年末持股参考线
        if not pd.isna(prev_share):
            ax.hlines(prev_share, *date_range, 
                     colors='grey', linestyles='--', 
                     label=f'{year-1}年末持股')
            
        # 标注当年减持比例
        if not pd.isna(total_reduction):
            ax.annotate(f'减持:{total_reduction:.2%}', 
                        (date_range[1], year_data['持股比例'].iloc[-1]),
                        textcoords="offset points",
                        xytext=(10,-10),
                        arrowprops=dict(arrowstyle="->"))
            
        # 标记约束状态（修正逻辑判断）
        if constraint is None:
            ax.axvspan(*date_range, alpha=0.1, color='blue')
            ax.text(date_range[0], ax.get_ylim()[1]*0.95,
                    '约束未定义', color='blue', weight='bold')
        elif constraint is False:  # 使用明确的条件判断
            ax.axvspan(*date_range, alpha=0.1, color='red')
            ax.text(date_range[0], ax.get_ylim()[1]*0.95,
                    '减持超限', color='red', weight='bold')
        else:
            ax.axvspan(*date_range, alpha=0.1, color='green')
            ax.text(date_range[0], ax.get_ylim()[1]*0.95,
                    '减持未超限', color='green', weight='bold')

    # 图表装饰
    plt.title(f'{company} - {shareholder} 持股变动分析')
    plt.xlabel('日期')
    plt.ylabel('持股比例')
    plt.grid(True)
    
    # 智能图例处理
    handles, labels = ax.get_legend_handles_labels()
    seen = []
    unique = [h for h, l in zip(handles, labels) 
             if not (l in seen or seen.append(l))]
    ax.legend(unique, [l for l in labels if l not in seen[:-1]])
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# df.to_csv("temp_meu_17_1.csv", encoding='utf-8-sig', index=False)
df[['公司简称', '股东', '日期', '持股比例', '当日减持比例', 'constraint']]

In [ ]:
# df[df['constraint']==False]

In [ ]:
# import matplotlib.pyplot as plt
# import pandas as pd

# def plot_meu_17_1_compliance(checked_df):
#     """
#     根据合规检查结果绘制分组趋势图：
#     - 按 [公司简称, 股东] 分组
#     - X轴为年份
#     - 显示以下指标：
#         1. 上年末基数（持股比例基准）
#         2. 当年累计减持比例
#         3. 违规情况标记
#     """
#     # 数据预处理
#     df = checked_df.copy()

    
#     # 按分组聚合关键指标
#     grouped = df.groupby(['公司简称', '股东', 'year']).apply(
#         lambda x: pd.Series({
#             '上年基数': x['持股比例_base'].iloc[0] if not x['持股比例_base'].isna().all() else None,
#             '当年减持总量': x['当日减持比例'].sum(),
#             '是否违规': (~x['meu_17_1_constraint']).any()
#         })
#     ).reset_index().dropna(subset=['上年基数'])  # 过滤无基准数据的记录

#     # 设置绘图风格
#     plt.style.use('seaborn')
    
#     # 为每个分组单独绘图
#     for (company, holder), group in grouped.groupby(['公司简称', '股东']):
#         fig, ax = plt.subplots(figsize=(12, 6))
        
#         # 排序确保时间序列正确
#         group = group.sort_values('year')
        
#         # 绘制柱状图
#         bars = ax.bar(
#             x=group['year'].astype(str),  # 将年份作为分类数据
#             height=group['当年减持总量'],
#             bottom=group['上年基数'],
#             label='当年减持量',
#             alpha=0.7
#         )
        
#         # 绘制基准线
#         ax.bar(
#             x=group['year'].astype(str),
#             height=group['上年基数'],
#             label='上年基数',
#             alpha=0.4
#         )
        
#         # 标注违规点
#         violation_years = group[group['是否违规']]['year']
#         for year in violation_years:
#             idx = group[group['year'] == year].index[0]
#             y_position = group.loc[idx, '上年基数'] + group.loc[idx, '当年减持总量']
#             ax.scatter(
#                 x=str(year),
#                 y=y_position * 1.05,  # 上浮5%避免重叠
#                 color='red',
#                 marker='X',
#                 s=120,
#                 zorder=10,
#                 label='违规' if idx == group.index[0] else ""
#             )
        
#         # 添加数值标签
#         for bar in bars:
#             height = bar.get_height()
#             bottom = bar.get_y() + bar.get_height()
#             ax.text(bar.get_x() + bar.get_width()/2., bottom * 1.02,
#                     f'{height:.2%}',
#                     ha='center', va='bottom')
        
#         # 图表装饰
#         ax.set_title(f"{company} - {holder} 持股变动合规分析", fontsize=14, pad=20)
#         ax.set_xlabel("会计年度", fontsize=12)
#         ax.set_ylabel("持股比例", fontsize=12)
#         ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
#         ax.grid(axis='y', linestyle='--', alpha=0.7)
        
#         plt.tight_layout()
#         plt.show()

# plot_meu_17_1_compliance(df)

---

## MEU_17_2


| 字段 | 内容 |
|------|------|
| subject | 上市公司董监高 |
| condition | 所持股份不超过1000股 |
| constrain | 可一次全部转让且不受前款转让比例限制 |
| contextual_info | nan |
| note | nan |
| relation | exclude |
| target | MEU_17_1 |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1701 |
| completion_tokens | 2813 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_17_2(df):
    '''
    检查MEU_17_2合规性：
    subject: 上市公司董监高
    condition: 所持股份不超过1000股
    constraint: 可一次全部转让且不受前款转让比例限制
    '''
    df = df.copy()
    
    # 初始化标记列
    df['meu_17_2_subject'] = False
    df['meu_17_2_condition'] = False
    df['meu_17_2_constraint'] = None  # 初始化为None便于布尔值填充
    
    # 1. 验证责任主体：上市公司董监高
    # 根据股东身份字段直接匹配
    valid_subject = df['股东身份'] == '董监高'
    df.loc[valid_subject, 'meu_17_2_subject'] = True
    
    # 2. 验证触发条件：持股数量≤1000股
    # 直接比较数值型字段
    valid_condition = df['持股数量'] <= 1000
    df.loc[valid_condition, 'meu_17_2_condition'] = True
    
    # 3. 验证约束条件：保留为None
    df['meu_17_2_constraint'] = None
    
    return df

In [ ]:
df = check_meu_17_2(df)
df

---

# Law Article 23

## MEU_23_1


| 字段 | 内容 |
|------|------|
| subject | 上市公司大股东 | 董监高 |
| condition | nan |
| constrain | 不得融券卖出本公司股份 |
| contextual_info | nan |
| note | nan |
| relation | nan |
| target | nan |
| type | 实际执行单元 |
| comments | nan |
| prompt_tokens | 1693 |
| completion_tokens | 6991 |


### 代码实现

In [ ]:
# 读取模拟数据
import pandas as pd
df = pd.read_csv('data_simulation/data_generated/data_simulate_08.csv')
date_columns = ['日期', '上市日期', '公告日期', '计划披露日', '计划开始日', '计划结束日', '离任日期']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
import pandas as pd

def check_meu_23_1(df):
    '''
    检查MEU_23_1合规性：
    "subject": "上市公司大股东 | 董监高",
    "condition": NaN,
    "constraint": "不得融券卖出本公司股份",
    "contextual_info": NaN
    '''
    df = df.copy()

    # 初始化标记列
    df['meu_23_1_subject'] = False
    df['meu_23_1_condition'] = False  # 初始化为False，后续统一处理
    df['meu_23_1_constraint'] = None  # 初始化为None，后续用布尔值填充

    # 1. 标记valid的subject（大股东或董监高）
    # 判断逻辑：股东身份属于董监高/控股股东/实际控制人，或持股比例≥5%
    subject_criteria = (
        df['股东身份'].isin(['董监高', '控股股东', '实际控制人']) |
        (df['持股比例'] >= 0.05)
    )
    df.loc[subject_criteria, 'meu_23_1_subject'] = True

    # 2. 标记valid的condition（由于MEU中condition为NaN，视为无条件触发，所有行标记为True）
    df['meu_23_1_condition'] = True  # 无条件约束，所有行条件满足

    # 3. 标记valid的constraint（减持方式非融券卖出）
    # 独立检查：无论subject是否满足，只要减持方式是融券卖出即违规
    constraint_met = (df['减持方式'] != '融券卖出')
    df['meu_23_1_constraint'] = constraint_met  # True表示合规，False表示违规

    return df


In [ ]:
df = check_meu_23_1(df)
df

---

In [ ]:
# import pandas as pd

# def check_meu_17_1(df):
#     '''
#     验证MEU_17_1合规性：
#     subject: 上市公司董监高 | 离任六个月内的董监高
#     condition: 通过集中竞价、大宗交易、协议转让等方式转让股份且不属于因司法强制执行、继承、遗赠、依法分割财产等导致股份变动
#     constraint: 每年转让的股份不得超过其所持本公司股份总数的25%
#     contextual_info: 以上年末其所持股总数作为基数计算可转让股份数量
#     '''
#     df = df.copy()

#     # 初始化标记列
#     df['meu_17_1_subject'] = False
#     df['meu_17_1_condition'] = False
#     df['meu_17_1_constraint'] = None

#     # ===================== 预处理 =====================
#     # 转换日期格式并提取年份
#     df['日期'] = pd.to_datetime(df['日期'])
#     df['离任日期'] = pd.to_datetime(df['离任日期'])
#     df['year'] = df['日期'].dt.year  # 新增全局年份列

#     # ===================== 标记subject =====================
#     # 时间条件（任期内或离任后6个月）
#     time_cond = (
#         df['离任日期'].isna() |  # 在任情况
#         (df['日期'] <= df['离任日期'] + pd.DateOffset(months=6))  # 离任后6个月
#     )
#     # 主体条件
#     valid_subject = (df['股东身份'] == '董监高') & time_cond
#     df.loc[valid_subject, 'meu_17_1_subject'] = True

#     # ===================== 标记condition =====================
#     # 处理所有数据（不再筛选subject）
#     df_sorted = df.sort_values(['公司简称', '股东', '日期']).copy()
    
#     # 计算减持比例
#     df_sorted['prev_持股比例'] = df_sorted.groupby(['公司简称', '股东'])['持股比例'].shift(1)
#     df_sorted['当日减持比例'] = (df_sorted['prev_持股比例'] - df_sorted['持股比例']).clip(lower=0)
    
#     # 条件检查（保留注释的代码）
#     method_cond = df_sorted['减持方式'].isin(['竞价交易', '大宗交易', '协议转让'])
#     # reason_cond = ~df_sorted['拟减持原因'].isin(['司法强制执行', '继承', '遗赠', '依法分割财产'])
#     has_daily_reduction = (df_sorted['当日减持比例'] > 0)
    
#     # 创建中间条件列
#     df_sorted['condition_met'] = has_daily_reduction & method_cond # & reason_cond
    
#     # 按分组标记condition（所有数据参与计算）
#     df_sorted['group_condition'] = df_sorted.groupby(
#         ['公司简称', '股东', 'year']
#     )['condition_met'].transform('any')

#     # 回写结果到原DF
#     df.loc[df_sorted.index, 'meu_17_1_condition'] = df_sorted['group_condition']

#     # ===================== 标记constraint =====================
#     # 获取上年末基准数据（使用全局year列）
#     year_end = df.groupby(['公司简称', '股东', 'year'])['日期'].idxmax()
#     base_df = df.loc[year_end, ['公司简称', '股东', 'year', '持股比例']].reset_index()
#     base_df['base_year'] = base_df['year'] + 1
    
#     # 合并基准数据（仅处理符合subject条件的记录）
#     df_constraint = df[valid_subject].merge(
#         base_df[['公司简称', '股东', 'base_year', '持股比例']],
#         left_on=['公司简称', '股东', 'year'],
#         right_on=['公司简称', '股东', 'base_year'],
#         how='left',
#         suffixes=('', '_base')
#     )
    
#     # 计算年度总减持比例
#     if not df_constraint.empty:
#         yearly_total = df_constraint.groupby(
#             ['公司简称', '股东', 'year']
#         )['当日减持比例'].transform('sum')
        
#         # 验证约束条件
#         valid_constraint = (yearly_total <= df_constraint['持股比例_base'] * 0.25) & df_constraint['持股比例_base'].notna()
#         df.loc[df_constraint.index, 'meu_17_1_constraint'] = valid_constraint

#     # # ===================== 清理中间列 =====================
#     # cols_to_drop = ['prev_持股比例', '当日减持比例', 'year', '持股数量_base', 'base_year',
#     #                'group_condition', 'condition_met']
#     # df.drop([col for col in cols_to_drop if col in df.columns], axis=1, inplace=True)

#     return df
